# Gold_Standard_Segmentation_v1.0

**Goal.** Produce a human-verified gold-standard segmentation of the large-FOV
hyperstack, to be used as the validation set for **Vulcan 1.1**. This notebook
does not train anything and does not run the model — it produces the ground
truth that the model is scored *against*, plus the scoring code itself.

**What is inherited**

| From | What |
|---|---|
| `Label_Generation_QC_v1.2` | histogram-isolation contract (`extract_plane` returns a copy), NPC-watershed droplet detection, nucleus-boundary-anchored NPC puncta, 4-channel multi-label representation |
| `Large_FOV_Nuclear_Pipeline_v16.2` | run-scoped path resolver, config-hash run ids, stale-guard, µm↔px conversion discipline, Stage-2 NPC-shell / membrane gate, per-instance labelling |

**What is new (§5)** — the nucleus detector is replaced.

---

## The failure this notebook is built to fix

Stated symptom: *small interior regions of a nucleus are labelled as separate
nuclei because they are locally bright relative to the nuclear interior.*

That is a **threshold** failure, not a **separation** failure, and the
distinction decides the fix:

* `detect_nucleus_adaptive` uses `threshold_local(nls, block_size ≈ D_droplet/3)`.
  For a nucleus occupying roughly half the droplet diameter, the block is the
  same order as the nucleus itself, so the local threshold **tracks the
  nucleus's own brightness**. Interior chromatin/NLS-dense patches clear it;
  the dimmer bulk of the interior does not. One nucleus → several blobs.
* `_clean_and_gate` then keeps only the largest connected component, so the
  surviving "nucleus" is a *fragment* — under-measured area, and in the U-Net
  path each fragment becomes its own instance under `measure.label`.

Consequences that matter downstream: nuclear area is biased low, N/C ratio is
biased high (the mask sits on the brightest interior pixels), and the
instance count per droplet is inflated.

### Why plain watershed is not the answer, and what is

Seeding a watershed on intensity maxima reproduces the same failure — every
bright interior patch is a local maximum and seeds its own basin. Measured on
the synthetic droplet of §7, a gradient watershed seeded that way scored
**Dice 0.36** — *worse* than the threshold baseline it was meant to replace.

The step that makes it work is **flattening the interior before seeding**:

1. **Opening-then-closing by morphological reconstruction** with a structuring
   element larger than the interior features but smaller than the nucleus.
   This removes bright domes and dark holes *smaller than the SE* while
   leaving the nuclear envelope step untouched — it is shape-selective, unlike
   Gaussian blur, which attenuates the envelope too.
2. **h-maxima seeding** on the flattened image. Every maximum within `h` of the
   dominant one is merged into a single seed, so an interior with residual
   texture still yields one seed per nucleus.
3. **Marker watershed on the gradient**, with a second marker on the droplet
   wall. The basin floods outward from the seed and stops at the strongest
   barrier on the path — the envelope — regardless of how heterogeneous the
   interior was. The result is one filled, connected region per seed **by
   construction**; fragmentation is not something that has to be repaired
   afterwards, it cannot occur.

Synthetic benchmark (§7, two contrast regimes × 8 droplets, measured):

| Regime | legacy Dice | legacy area / truth | new Dice | new area / truth |
|---|---|---|---|---|
| bright uniform nucleus | 0.96 | 0.93 | **0.99** | 0.98 |
| dim nucleus, bright interior patches | 0.49 | **0.34** | **0.98** | 0.98 |

The second row is the reported symptom, reproduced: the legacy detector keeps
**a third of the nucleus**. The first row is the honest control — where the
interior is uniform enough, the legacy Otsu fallback is fine and the new
method buys little. The fix matters exactly where the interior is
heterogeneous, which is where the complaint came from.

### The risk this introduces, and the gate that covers it

A marker watershed **always returns a region for every seed**. On a droplet
with no nucleus (t = 0–2) it floods to the wall — measured at 78 % of droplet
area on the synthetic no-nucleus control. Thresholding could return nothing;
watershed cannot. Acceptance is therefore *not* optional here, and §6 applies
three independent tests before a mask is kept:

| Test | Rejects |
|---|---|
| droplet-area fraction ∈ [0.01, 0.50] | wall floods, whole-droplet false positives |
| NE contrast: mean NLS inside / mean in an outside shell ≥ 1.05 | seeds on cytoplasmic noise |
| Stage-2 NPC-shell + membrane co-localisation (ported from v16.2) | droplet caps |

---

## Gold-standard requirements

1. **Sampled, not exhaustive.** A stratified, seeded sample of planes (§9), so
   the set is defensible and re-derivable rather than "whatever we annotated".
2. **Human-verified.** Automatic output is a *proposal*; napari review (§11)
   is what makes it ground truth.
3. **Explicit ignorance.** `255 = UNANNOTATED` survives into the saved labels
   and is honoured as an ignore-region by the metrics (§13) — never silently
   treated as background.
4. **Traceable.** Every sample carries the config hash, pixel size, z-step and
   notebook version. Two pixel-size bugs have already invalidated
   measurements on this dataset; the gold standard must not be the third.

**Conventions.** Dim order `(T, Z, C, Y, X)`; channels `0 = Membrane,
1 = NLS, 2 = NPC`; classes `0 = Background, 1 = Droplet, 2 = NPC,
3 = Nucleus`, multi-label (a nucleus is *also* droplet).

## 1. Environment and imports

In [ ]:
from __future__ import annotations

import getpass
import hashlib
import json
import os
import sys
import warnings
from dataclasses import dataclass, asdict, field
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from scipy import ndimage as ndi
from skimage import filters, measure, morphology, segmentation

try:
    import tifffile as tiff
except Exception:                                    # pragma: no cover
    tiff = None
    warnings.warn("tifffile unavailable — real-data cells will not run.")

RNG_SEED = 20260825
rng = np.random.default_rng(RNG_SEED)

print("numpy      ", np.__version__)
import skimage as _sk; print("scikit-image", _sk.__version__)
print("seed       ", RNG_SEED)

## 2. Configuration

**Everything physical is declared in µm and converted to px through
`pixel_size_um`.** The legacy notebooks hard-coded pixel values that were
tuned by eye on this dataset while `pixel_size_um` was wrong (0.108, then
0.2167; true value 0.1625). The tuning was still valid *in pixels* — it was
done on real images — so the µm defaults below are the tuned pixel values
re-expressed at the **correct** scale, not the µm numbers that appeared in
the old configs.

The one that changed meaning is the droplet area floor:

| | v7 intent | v7 actual behaviour |
|---|---|---|
| `MIN_DROPLET_AREA` | 150 µm² | `um2_to_px(150)` at 0.108 = **12 860 px** = **340 µm²** at the true scale |

So the floor that was visually validated is ~340 µm², not 150. The default
preserves the validated behaviour; `droplet_min_area_um2 = 150.0` would
quietly admit droplets v7 never accepted.

In [ ]:
@dataclass
class GoldStandardConfig:
    # ── provenance ────────────────────────────────────────────────────────
    notebook_version: str = "gold_standard_v1.0"
    dataset_name: str = "control_extract_1.1.tif"
    model_name: str = "Vulcan_1.1_best.keras"
    rig_name: str = "ix85_spin"

    # ── acquisition (see rigs/ix85_spin.json) ────────────────────────────
    pixel_size_um: float = 0.1625        # 6.5 µm sensor / 40x — optically + geometrically confirmed
    z_step_um: float = 2.0
    ch_membrane: int = 0
    ch_nls: int = 1
    ch_npc: int = 2

    # ── droplet detection (ported from Label_Generation_QC v1.2) ─────────
    droplet_clip_lo_pct: float = 1.0
    droplet_clip_hi_pct: float = 80.0
    droplet_blur_um: float = 1.30        # v7 sigma 8 px
    droplet_block_um: float = 48.91      # v7 block 301 px
    droplet_offset: float = -0.05
    droplet_min_sep_um: float = 6.50     # v7 min_distance 40 px
    droplet_min_area_um2: float = 340.0  # v7 12 860 px at the TRUE pixel size
    droplet_max_area_um2: float = 1500.0
    droplet_min_circ: float = 0.70
    droplet_erode_um: float = 1.63       # v7 10 px

    # ── nucleus: flattened seeded watershed (NEW — §5) ───────────────────
    nuc_smooth_um: float = 0.40          # pre-smoothing, kills detector/shot noise only
    nuc_flatten_um: float = 4.00         # SE radius: > interior features, < nuclear radius
    # DO NOT re-enable a droplet-relative clamp here. Scaling the SE with
    # droplet radius makes measured nuclear area a function of droplet size —
    # measured at -54% in a 200 um2 droplet vs -2% above 500 um2, i.e. a
    # manufactured nuclear-size/droplet-size correlation, which is the effect
    # this project exists to measure. See the guard test in section 7c.
    nuc_flatten_max_r_frac: Optional[float] = None
    nuc_h_frac: float = 0.35             # h-maxima depth, as a fraction of (p99.5 - median)
    nuc_seed_min_area_um2: float = 3.0
    nuc_max_seeds: int = 1               # one nucleus per droplet in this system
    nuc_wall_standoff_um: float = 1.50   # width of the droplet-wall background marker
    nuc_edge_smooth_um: float = 0.0      # extra smoothing before the gradient; 0 = off

    # ── nucleus acceptance gates (§6) ────────────────────────────────────
    nuc_min_frac_of_droplet: float = 0.01
    nuc_max_frac_of_droplet: float = 0.50
    nuc_min_solidity: float = 0.80
    nuc_ne_contrast_min: float = 1.05    # mean NLS inside / mean in outside shell
    nuc_ne_shell_um: float = 1.30
    require_stage2_gate: bool = True

    # ── Stage-2 gate (ported verbatim from v16.2 §7a) ────────────────────
    npc_margin_um: float = 0.81          # v16.2 npc_margin_px 5
    npc_std_mult: float = 2.0
    gate_min_puncta: int = 8
    gate_shell_tol_um: float = 0.98      # v16.2 6 px
    gate_shell_min_inlier_frac: float = 0.45
    gate_puncta_min_size_um2: float = 0.16
    gate_min_ring_contrast: Optional[float] = 1.15
    gate_membrane_nn_radius_um: float = 0.98
    gate_membrane_min_coloc_frac: float = 0.30
    gate_membrane_k_std: float = 1.0
    ransac_n_iter: int = 200
    gate_min_timepoint: Optional[int] = 2   # t = 0-1 has too few puncta to fit a shell

    # ── gold-standard sampling (§9) ──────────────────────────────────────
    sample_seed: int = RNG_SEED
    sample_planes_per_timepoint: int = 2
    sample_z_floor: int = 6              # coverslip artefacts below this z
    sample_z_ceiling: Optional[int] = None
    sample_max_droplets_per_plane: int = 40
    patch_size: int = 512

    # ── evaluation (§13) ─────────────────────────────────────────────────
    eval_match_iou: float = 0.50         # IoU for a true positive
    eval_touch_iou: float = 0.10         # IoU for "these two objects overlap at all"

    # ── derived: µm → px ─────────────────────────────────────────────────
    def um(self, v: float) -> float:
        """µm → px (float)."""
        return v / self.pixel_size_um

    def px(self, v: float) -> int:
        """µm → px (int, min 1)."""
        return max(int(round(v / self.pixel_size_um)), 1)

    def um2_to_px(self, a: float) -> float:
        return a / (self.pixel_size_um ** 2)

    def px_to_um2(self, a: float) -> float:
        return a * (self.pixel_size_um ** 2)

    @property
    def droplet_block_px(self) -> int:
        b = self.px(self.droplet_block_um)
        return b + 1 if b % 2 == 0 else b          # threshold_local needs odd

    def to_signature(self) -> Dict[str, object]:
        d = asdict(self)
        d.pop("notebook_version", None)
        return d

    def run_id(self) -> str:
        h = hashlib.sha1(json.dumps(self.to_signature(), sort_keys=True,
                                    default=str).encode()).hexdigest()[:8]
        return f"gold_standard__{Path(self.dataset_name).stem}__cfg-{h}"


cfg = GoldStandardConfig()

CLASS_BACKGROUND, CLASS_DROPLET, CLASS_NPC, CLASS_NUCLEUS = 0, 1, 2, 3
CLASS_NAMES = ["Background", "Droplet", "NPC", "Nucleus"]
UNANNOTATED = 255

print(cfg.run_id())
print(f"droplet: blur {cfg.um(cfg.droplet_blur_um):.1f}px  "
      f"block {cfg.droplet_block_px}px  "
      f"min_sep {cfg.um(cfg.droplet_min_sep_um):.0f}px  "
      f"area {cfg.um2_to_px(cfg.droplet_min_area_um2):.0f}-"
      f"{cfg.um2_to_px(cfg.droplet_max_area_um2):.0f}px")
print(f"nucleus: smooth {cfg.um(cfg.nuc_smooth_um):.1f}px  "
      f"flatten SE r={cfg.px(cfg.nuc_flatten_um)}px  "
      f"wall standoff {cfg.px(cfg.nuc_wall_standoff_um)}px")

## 2b. Paths and run bootstrap

Same two-root layout as v16.2: code under `$HOME/Projects`, data under
`/data/user/$USER/Nuclear_Scaling`. Outputs are isolated under
`Runs/<run_id>/`, so a config change cannot silently mix with an earlier
gold-standard build. Falls back to a local directory off Cheaha.

In [ ]:
GS_SUBDIRS = ("proposals", "review", "committed", "qc", "eval")


class GoldStandardPaths:
    def __init__(self, cfg: GoldStandardConfig, data_root: Optional[Path] = None):
        user = getpass.getuser()
        if data_root is None:
            env = os.environ.get("USER_DATA")
            base = Path(env) if env else Path(f"/data/user/{user}")
            if not base.exists():                    # laptop / desktop fallback
                base = Path.home() / "Data"
            data_root = base / "Nuclear_Scaling"
        self.data_root = Path(data_root).expanduser()
        self.raw_images_dir = self.data_root / "Inputs" / "Raw_Images"
        self.models_dir = self.data_root / "Inputs" / "Models"
        self.run_dir = self.data_root / "Runs" / cfg.run_id()
        for s in GS_SUBDIRS:
            setattr(self, f"{s}_dir", self.run_dir / s)

    def make_all(self) -> "GoldStandardPaths":
        for s in GS_SUBDIRS:
            getattr(self, f"{s}_dir").mkdir(parents=True, exist_ok=True)
        return self

    @property
    def image_path(self) -> Path:
        return self.raw_images_dir / cfg.dataset_name

    @property
    def model_path(self) -> Path:
        return self.models_dir / cfg.model_name

    @property
    def config_snapshot(self) -> Path:
        return self.run_dir / "config.json"

    def snapshot(self, cfg: GoldStandardConfig) -> None:
        payload = {"config": asdict(cfg),
                   "written_at": datetime.now().isoformat(timespec="seconds"),
                   "run_id": cfg.run_id()}
        self.config_snapshot.write_text(json.dumps(payload, indent=2, default=str))

    def assert_config_matches(self, cfg: GoldStandardConfig, strict: bool = True) -> None:
        """Refuse to extend a gold-standard set that was built under a different config."""
        if not self.config_snapshot.exists():
            return
        snap = json.loads(self.config_snapshot.read_text())["config"]
        live = asdict(cfg)
        diffs = {k: (snap.get(k), live.get(k)) for k in set(snap) | set(live)
                 if snap.get(k) != live.get(k)}
        if diffs:
            msg = "[stale-guard] live config differs from the run snapshot:\n" + "\n".join(
                f"    {k}: snapshot={s!r} live={l!r}" for k, (s, l) in diffs.items())
            if strict:
                raise RuntimeError(msg)
            print("WARNING:", msg)


gs_paths = GoldStandardPaths(cfg).make_all()
gs_paths.assert_config_matches(cfg)
gs_paths.snapshot(cfg)
print("run dir:", gs_paths.run_dir)
print("image  :", gs_paths.image_path, "|exists:", gs_paths.image_path.exists())

## 3. Histogram-safe data access

The v1.2 contract, unchanged: every channel read returns a fresh `float32`
**copy**, so no downstream step can clip the source in place. The NPC channel
is read twice with opposite requirements — clipped for droplet detection
(puncta suppressed so they cannot fragment the watershed), raw for puncta
detection (puncta are the signal) — and sharing one array between them
silently destroys the NPC class.

In [ ]:
def load_hyperstack(path: Path):
    """Memory-map (T, Z, C, Y, X); fall back to a full read for compressed TIFFs."""
    if tiff is None:
        raise ImportError("tifffile is required.")
    try:
        hs = tiff.memmap(str(path), mode="r")
        print(f"memmapped {hs.shape} {hs.dtype}")
    except (ValueError, MemoryError, NotImplementedError) as e:
        print(f"memmap unavailable ({e}); reading fully")
        hs = tiff.imread(str(path))
        print(f"loaded {hs.shape} {hs.dtype}")
    if hs.ndim != 5:
        raise ValueError(f"expected 5-D (T,Z,C,Y,X), got {hs.shape}")
    return hs


def extract_plane(hyperstack, t: int, z: int, c: int) -> np.ndarray:
    """One 2-D plane as a fresh float32 COPY. Never a view."""
    return np.asarray(hyperstack[t, z, c], dtype=np.float32).copy()


def clip_histogram(img: np.ndarray, lo_pct: float, hi_pct: float) -> np.ndarray:
    """Percentile-clip and rescale a COPY to [0, 1]."""
    work = img.astype(np.float32, copy=True)
    lo, hi = np.percentile(work, [lo_pct, hi_pct])
    if hi <= lo:
        return np.zeros_like(work)
    return (np.clip(work, lo, hi) - lo) / (hi - lo)


def assert_histogram_isolation(hyperstack, t: int, z: int) -> None:
    """Prove a full detection pass leaves the source channel byte-identical."""
    before = np.asarray(hyperstack[t, z, cfg.ch_npc]).copy()
    _ = detect_droplets(extract_plane(hyperstack, t, z, cfg.ch_npc), cfg)
    after = np.asarray(hyperstack[t, z, cfg.ch_npc])
    assert np.array_equal(before, after), "NPC channel was mutated in place"
    print("histogram isolation OK")


def crop_bbox(arr: np.ndarray, bbox: Tuple[int, int, int, int]) -> np.ndarray:
    r0, c0, r1, c1 = bbox
    return arr[r0:r1, c0:c1]


def _circularity(region) -> float:
    p = region.perimeter
    return float(4.0 * np.pi * region.area / (p * p)) if p > 0 else 0.0

## 4. Droplet detection

Ported from `Label_Generation_QC` §Stage 1 with pixel constants replaced by
µm-derived ones. Seeds come from the **distance transform of the binary
foreground** — shape, never intensity — so an internal NPC ring cannot seed a
basin. An upper area gate is added: v7 had only a floor, so a merged
multi-droplet blob that happened to be round enough passed.

In [ ]:
def detect_droplets(npc_plane: np.ndarray, cfg: GoldStandardConfig) -> List[dict]:
    """
    Detect droplets from one NPC plane.

    1. clip p1-p80 on a copy      → bright puncta suppressed
    2. Gaussian blur              → droplet-scale structure only
    3. local adaptive threshold   → binary foreground
    4. fill holes, drop debris
    5. distance transform → peak_local_max seeds (shape-based)
    6. watershed on -distance, masked to foreground
    7. per region: fill, area + circularity gates, erode inward

    Returns list of {label, mask, bbox, centroid, area_px, area_um2, circ}.
    """
    from skimage.feature import peak_local_max

    img = clip_histogram(npc_plane, cfg.droplet_clip_lo_pct, cfg.droplet_clip_hi_pct)
    img = filters.gaussian(img, sigma=cfg.um(cfg.droplet_blur_um), preserve_range=True)

    thr = filters.threshold_local(img, block_size=cfg.droplet_block_px,
                                  offset=cfg.droplet_offset)
    fg = img > thr
    fg = ndi.binary_fill_holes(fg)
    min_area = int(cfg.um2_to_px(cfg.droplet_min_area_um2))
    max_area = int(cfg.um2_to_px(cfg.droplet_max_area_um2))
    fg = morphology.remove_small_objects(fg, min_size=min_area)
    if not fg.any():
        return []

    dist = ndi.distance_transform_edt(fg)
    seeds = peak_local_max(dist, min_distance=cfg.px(cfg.droplet_min_sep_um), labels=fg)
    if len(seeds) == 0:
        return []
    markers = np.zeros(dist.shape, np.int32)
    markers[tuple(seeds.T)] = np.arange(1, len(seeds) + 1)
    ws = segmentation.watershed(-dist, markers, mask=fg)

    selem = morphology.disk(cfg.px(cfg.droplet_erode_um))
    droplets: List[dict] = []
    for region in measure.regionprops(ws):
        if not (min_area <= region.area <= max_area):
            continue
        circ = _circularity(region)
        if circ < cfg.droplet_min_circ:
            continue                                   # merged / kidney-bean
        solid = ndi.binary_fill_holes(ws == region.label)
        eroded = morphology.binary_erosion(solid, selem)
        if not eroded.any():
            continue
        ys, xs = np.where(eroded)
        droplets.append({
            "label": int(region.label),
            "mask": eroded,
            "bbox": (int(ys.min()), int(xs.min()), int(ys.max()) + 1, int(xs.max()) + 1),
            "centroid": (float(ys.mean()), float(xs.mean())),
            "area_px": int(eroded.sum()),
            "area_um2": float(cfg.px_to_um2(int(eroded.sum()))),
            "circ": round(circ, 3),
        })
    droplets.sort(key=lambda d: d["area_px"], reverse=True)
    return droplets


def diagnose_droplet_detection(npc_plane: np.ndarray, cfg: GoldStandardConfig
                               ) -> pd.DataFrame:
    """
    Gate-by-gate attrition for detect_droplets. Run this whenever a plane
    returns zero droplets — it says WHICH stage dropped them, which is the
    difference between a threshold problem and an area-window problem.
    """
    from skimage.feature import peak_local_max

    img = clip_histogram(npc_plane, cfg.droplet_clip_lo_pct, cfg.droplet_clip_hi_pct)
    img = filters.gaussian(img, sigma=cfg.um(cfg.droplet_blur_um), preserve_range=True)
    thr = filters.threshold_local(img, block_size=cfg.droplet_block_px,
                                  offset=cfg.droplet_offset)
    fg_raw = img > thr
    fg_filled = ndi.binary_fill_holes(fg_raw)
    min_area = int(cfg.um2_to_px(cfg.droplet_min_area_um2))
    max_area = int(cfg.um2_to_px(cfg.droplet_max_area_um2))
    fg = morphology.remove_small_objects(fg_filled, min_size=min_area)

    steps = [("foreground pixels after threshold", int(fg_raw.sum())),
             ("connected components before area filter",
              int(measure.label(fg_filled).max())),
             (f"components >= min area ({min_area} px / "
              f"{cfg.droplet_min_area_um2:.0f} um2)", int(measure.label(fg).max()))]

    n_seeds = n_ws = n_area = n_circ = n_eroded = 0
    areas, circs = [], []
    if fg.any():
        dist = ndi.distance_transform_edt(fg)
        seeds = peak_local_max(dist, min_distance=cfg.px(cfg.droplet_min_sep_um),
                               labels=fg)
        n_seeds = len(seeds)
        if n_seeds:
            markers = np.zeros(dist.shape, np.int32)
            markers[tuple(seeds.T)] = np.arange(1, n_seeds + 1)
            ws = segmentation.watershed(-dist, markers, mask=fg)
            selem = morphology.disk(cfg.px(cfg.droplet_erode_um))
            for region in measure.regionprops(ws):
                n_ws += 1
                areas.append(region.area)
                circs.append(_circularity(region))
                if not (min_area <= region.area <= max_area):
                    continue
                n_area += 1
                if circs[-1] < cfg.droplet_min_circ:
                    continue
                n_circ += 1
                solid = ndi.binary_fill_holes(ws == region.label)
                if morphology.binary_erosion(solid, selem).any():
                    n_eroded += 1

    steps += [(f"watershed seeds (min sep {cfg.px(cfg.droplet_min_sep_um)} px)", n_seeds),
              ("watershed regions", n_ws),
              (f"pass area window [{min_area}, {max_area}] px", n_area),
              (f"pass circularity >= {cfg.droplet_min_circ}", n_circ),
              (f"survive {cfg.px(cfg.droplet_erode_um)} px erosion", n_eroded)]

    df = pd.DataFrame(steps, columns=["stage", "count"])
    if areas:
        a = np.array(areas, float)
        print(f"  watershed region areas: median {np.median(a):.0f} px "
              f"({cfg.px_to_um2(np.median(a)):.0f} um2), "
              f"range {a.min():.0f}-{a.max():.0f} px "
              f"({cfg.px_to_um2(a.min()):.0f}-{cfg.px_to_um2(a.max()):.0f} um2)")
        print(f"  circularity: median {np.median(circs):.2f}, "
              f"max {np.max(circs):.2f}")
        below = int((a < min_area).sum()); above = int((a > max_area).sum())
        if below or above:
            print(f"  area window rejects {below} below and {above} above "
                  f"— compare the ranges above against "
                  f"[{cfg.droplet_min_area_um2:.0f}, {cfg.droplet_max_area_um2:.0f}] um2")
    return df

## 5. Nucleus segmentation — flattened seeded watershed  *(the new part)*

Three functions, each doing one thing, so each can be inspected in isolation:

| Function | Job | Failure it removes |
|---|---|---|
| `flatten_interior` | reconstruction opening+closing at SE radius `nuc_flatten_um` | interior bright patches and dark holes stop being edges |
| `seed_nucleus` | h-maxima on the flattened image, keep `nuc_max_seeds` strongest | many maxima → one seed |
| `segment_nucleus_watershed` | marker watershed on the gradient, seed vs. droplet wall | mask extent stops at the envelope, not at a patch edge |

**Why reconstruction and not a Gaussian.** A Gaussian attenuates *everything*
at the smoothing scale, including the nuclear envelope, so a blur strong enough
to erase a 2 µm chromatin patch also softens the boundary you are trying to
find. Reconstruction is shape-selective: opening by reconstruction removes
bright structures that do not contain the structuring element, and then
*restores the surviving structures to their original shape* by geodesic
dilation. The envelope step is preserved exactly.

**Choosing `nuc_flatten_um`.** It must be larger than the interior features to
suppress and smaller than the nuclear radius, and it is the one parameter here
with real leverage. Measured on the heterogeneous regime of §7:
3.0 µm → Dice 0.66, 4.0 µm → **0.98**, 5–6 µm → 0.98 (flat). On the uniform
regime the same change is invisible (0.987 → 0.991), so a larger SE costs
nothing there. The upper limit is set by the nucleus: at r_nucleus = 3 µm a
4 µm SE erodes the nucleus itself (Dice 0.66). Real nuclei here run
100–600 µm², i.e. r = 5.6–13.8 µm, so 4.0 µm sits inside the working range —
but the margin at the small end is thin, which is why
`nuc_flatten_max_r_frac` clamps the SE relative to droplet size and why §14
flags a two-pass version as the fix if early nuclei come out poorly.

`nuc_h_frac` turned out to be insensitive (0.20 / 0.35 / 0.55 gave identical
Dice to 3 decimals across the sweep) — after flattening there is only one
maximum left to find, which is the point.

**What "cannot fragment" means.** `watershed(..., markers, mask=interior)`
assigns every masked pixel to exactly one marker basin. With one nucleus seed
the nucleus label is a single connected region by construction, then
hole-filled. There is no largest-component fallback, because there is never
more than one component to choose between.

In [ ]:
def flatten_interior(img_norm: np.ndarray, se_radius_px: int) -> np.ndarray:
    """
    Suppress bright domes and dark holes smaller than the structuring element,
    while preserving the shape and position of everything larger.

    opening-by-reconstruction  : erode, then geodesic-dilate back under the original
    closing-by-reconstruction  : dilate, then geodesic-erode back under that result

    Edges of surviving structures are NOT displaced — that is the property a
    Gaussian does not have and the reason this step is what fixes the problem.
    """
    se = morphology.disk(max(int(se_radius_px), 1))
    opened = morphology.reconstruction(morphology.erosion(img_norm, se),
                                       img_norm, method="dilation")
    closed = morphology.reconstruction(morphology.dilation(opened, se),
                                       opened, method="erosion")
    return closed.astype(np.float32)


def seed_nucleus(flat: np.ndarray, interior: np.ndarray, cfg: GoldStandardConfig
                 ) -> Tuple[np.ndarray, dict]:
    """
    h-maxima seeding. Every regional maximum within `h` of the dominant one is
    merged into a single seed, so residual interior texture cannot split it.

    h is scaled to the droplet's own dynamic range, (p99.5 - median) inside the
    droplet, so one setting works across timepoints as NLS accumulates.
    """
    vals = flat[interior]
    med = float(np.median(vals))
    top = float(np.percentile(vals, 99.5))
    h = cfg.nuc_h_frac * max(top - med, 1e-6)

    hmax = morphology.h_maxima(np.where(interior, flat, 0.0), h)
    seeds = (hmax > 0) & interior
    seeds = morphology.remove_small_objects(
        seeds, min_size=max(int(cfg.um2_to_px(cfg.nuc_seed_min_area_um2)), 1))

    info = {"h": h, "droplet_median": med, "droplet_p995": top, "n_seeds_raw": 0}
    if not seeds.any():
        return np.zeros_like(interior, np.int32), info

    lbl = measure.label(seeds)
    props = measure.regionprops(lbl, intensity_image=flat)
    info["n_seeds_raw"] = len(props)
    # rank by total excess signal, not area: a small very bright seed beats a
    # large dim one, which is the right preference for a nucleus vs. debris
    props.sort(key=lambda r: float(r.intensity_mean) * float(r.area), reverse=True)
    keep = [p.label for p in props[:max(cfg.nuc_max_seeds, 1)]]
    return measure.label(np.isin(lbl, keep)).astype(np.int32), info


def segment_nucleus_watershed(nls_crop: np.ndarray, droplet_mask_crop: np.ndarray,
                              cfg: GoldStandardConfig) -> Tuple[np.ndarray, dict]:
    """
    Segment the nucleus inside one droplet.

    Returns (nucleus_mask bool, info dict). `info["accepted"]` is False when a
    gate rejected the region; the mask is returned regardless so a reviewer can
    see WHAT was rejected, and §10 maps rejections to UNANNOTATED rather than
    to background.
    """
    interior = droplet_mask_crop.astype(bool)
    info: Dict[str, object] = {
        "method": "watershed", "accepted": False, "reject": "",
        "n_seeds_raw": 0, "h": np.nan, "area_px": 0, "area_um2": np.nan,
        "frac_of_droplet": np.nan, "solidity": np.nan,
        "ne_contrast": np.nan, "boundary_gradient": np.nan,
    }
    empty = np.zeros(interior.shape, bool)
    area_d = int(interior.sum())
    if area_d == 0:
        info["reject"] = "empty droplet mask"
        return empty, info

    # ── normalise WITHIN the droplet ─────────────────────────────────────
    # A global stretch makes `h` mean different things in a bright droplet and
    # a dim one; per-droplet normalisation is what lets one h_frac hold across
    # the whole timecourse.
    img = np.asarray(nls_crop, np.float32)
    lo, hi = np.percentile(img[interior], [1.0, 99.5])
    if hi <= lo:
        info["reject"] = "flat droplet interior"
        return empty, info
    norm = np.clip((img - lo) / (hi - lo), 0.0, 1.0)

    sm = filters.gaussian(norm, sigma=max(cfg.um(cfg.nuc_smooth_um), 0.5),
                          preserve_range=True).astype(np.float32)

    # ── flatten, with the SE clamped relative to droplet size ────────────
    se_px = cfg.px(cfg.nuc_flatten_um)
    if cfg.nuc_flatten_max_r_frac is not None:      # off by default — see config
        r_drop_px = np.sqrt(area_d / np.pi)
        se_px = int(min(se_px, max(int(cfg.nuc_flatten_max_r_frac * r_drop_px), 1)))
    flat = flatten_interior(sm, se_px)
    info["flatten_se_px"] = se_px

    # ── seed ─────────────────────────────────────────────────────────────
    seed_lbl, seed_info = seed_nucleus(flat, interior, cfg)
    info.update({k: seed_info[k] for k in ("h", "n_seeds_raw")})
    if seed_lbl.max() == 0:
        info["reject"] = "no seed survived"
        return empty, info

    # ── markers: 1 = droplet wall (background), 2.. = nucleus seeds ──────
    inner = morphology.binary_erosion(
        interior, morphology.disk(cfg.px(cfg.nuc_wall_standoff_um)))
    wall_ring = interior & ~inner
    if not wall_ring.any():
        wall_ring = interior & ~morphology.binary_erosion(interior, morphology.disk(1))

    markers = np.zeros(interior.shape, np.int32)
    markers[wall_ring] = 1
    markers[seed_lbl > 0] = seed_lbl[seed_lbl > 0] + 1

    surface = flat if cfg.nuc_edge_smooth_um <= 0 else filters.gaussian(
        flat, sigma=cfg.um(cfg.nuc_edge_smooth_um), preserve_range=True)
    grad = filters.sobel(surface)

    ws = segmentation.watershed(grad, markers, mask=interior)
    nucleus = ndi.binary_fill_holes(ws > 1)
    if not nucleus.any():
        info["reject"] = "empty basin"
        return empty, info

    accepted, gate_info = accept_nucleus(nucleus, img, interior, cfg)
    info.update(gate_info)
    info["accepted"] = accepted
    info["boundary_gradient"] = float(
        grad[nucleus & ~morphology.binary_erosion(nucleus, morphology.disk(1))].mean())
    return nucleus, info

## 6. Acceptance gates

A marker watershed always returns a region for every seed — on an empty
droplet it floods to the wall (measured: 78 % of droplet area, §7). Gating is
therefore load-bearing here in a way it was not for a thresholding detector,
which could simply return nothing.

**A rejected candidate is not background.** For a gold standard, labelling a
rejected-but-possibly-real nucleus as background would penalise Vulcan for
finding it. Rejections are written as `UNANNOTATED` (255) and excluded from
the metrics unless a reviewer resolves them (§10, §12, §13).

**This runs the strict Stage-2 setting, not generator parity.** v16.2 records
that the ported shell test is close to vacuous — its puncta zone is already a
10 px annulus and the RANSAC tolerance is 6 px, so on a flat NPC channel it
passes on noise alone. The two knobs v16.2 added and defaulted off
(`gate_puncta_min_size`, `gate_min_ring_contrast`) are **on** here. This is a
deliberate divergence: the training generator should be permissive, the gold
standard should not assert what it cannot verify.

In [ ]:
def ne_contrast(nucleus_mask: np.ndarray, nls_raw: np.ndarray,
                interior: np.ndarray, cfg: GoldStandardConfig) -> float:
    """
    Mean raw NLS inside the mask / mean in a shell just outside it, restricted
    to the droplet. Below ~1.0 the "nucleus" is not NLS-enriched at all.
    """
    shell_px = cfg.px(cfg.nuc_ne_shell_um)
    outside = (morphology.binary_dilation(nucleus_mask, morphology.disk(shell_px))
               & ~nucleus_mask & interior)
    if not nucleus_mask.any() or not outside.any():
        return float("nan")
    outer_mean = float(nls_raw[outside].mean())
    if outer_mean <= 0:
        return float("nan")
    return float(nls_raw[nucleus_mask].mean() / outer_mean)


def accept_nucleus(nucleus_mask: np.ndarray, nls_raw: np.ndarray,
                   interior: np.ndarray, cfg: GoldStandardConfig
                   ) -> Tuple[bool, dict]:
    """Geometry + NLS-enrichment gates. The Stage-2 gate is applied separately."""
    area = int(nucleus_mask.sum())
    area_d = int(interior.sum())
    frac = area / area_d if area_d else float("nan")
    rp = measure.regionprops(nucleus_mask.astype(np.uint8))
    solidity = float(rp[0].solidity) if rp else float("nan")
    contrast = ne_contrast(nucleus_mask, nls_raw, interior, cfg)

    info = {"area_px": area, "area_um2": float(cfg.px_to_um2(area)),
            "frac_of_droplet": float(frac), "solidity": solidity,
            "ne_contrast": contrast, "reject": ""}

    if not (cfg.nuc_min_frac_of_droplet <= frac <= cfg.nuc_max_frac_of_droplet):
        info["reject"] = (f"area frac {frac:.3f} outside "
                          f"[{cfg.nuc_min_frac_of_droplet}, {cfg.nuc_max_frac_of_droplet}]")
        return False, info
    if not np.isfinite(solidity) or solidity < cfg.nuc_min_solidity:
        info["reject"] = f"solidity {solidity:.3f} < {cfg.nuc_min_solidity}"
        return False, info
    if not np.isfinite(contrast) or contrast < cfg.nuc_ne_contrast_min:
        info["reject"] = f"NE contrast {contrast:.3f} < {cfg.nuc_ne_contrast_min}"
        return False, info
    return True, info

### 6b. Stage-2 gate — NPC shell + membrane co-localisation

Ported from `Large_FOV_Nuclear_Pipeline_v16.2` §7a with pixel constants
replaced by µm-derived ones. Asks the question the NLS channel cannot answer:
*is there an organised envelope around this thing?* A droplet cap fails it.
RANSAC is seeded at 0 so the verdict is reproducible — a gold standard that
changes between runs is not one.

In [ ]:
def _circle_from_3(p1, p2, p3):
    ax, ay = p1; bx, by = p2; cx, cy = p3
    d = 2.0 * (ax * (by - cy) + bx * (cy - ay) + cx * (ay - by))
    if abs(d) < 1e-9:
        return None
    a2, b2, c2 = ax * ax + ay * ay, bx * bx + by * by, cx * cx + cy * cy
    ux = (a2 * (by - cy) + b2 * (cy - ay) + c2 * (ay - by)) / d
    uy = (a2 * (cx - bx) + b2 * (ax - cx) + c2 * (bx - ax)) / d
    return ux, uy, float(np.hypot(ux - ax, uy - ay))


def _fit_circle_kasa(pts):
    x, y = pts[:, 0], pts[:, 1]
    A = np.c_[x, y, np.ones(len(x))]
    b = x * x + y * y
    sol, *_ = np.linalg.lstsq(A, b, rcond=None)
    cx, cy = sol[0] / 2.0, sol[1] / 2.0
    return cx, cy, np.sqrt(max(sol[2] + cx * cx + cy * cy, 0.0))


def fit_circle_ransac(pts, tol_px=4.0, n_iter=200, min_inlier_frac=0.5, rng=None):
    """RANSAC circle fit to Nx2 (x, y). Deterministic: rng defaults to seed 0."""
    rng = rng or np.random.default_rng(0)
    n = len(pts)
    if n < 3:
        return None
    best_inliers, best_circle = None, None
    for _ in range(n_iter):
        idx = rng.choice(n, 3, replace=False)
        circ = _circle_from_3(pts[idx[0]], pts[idx[1]], pts[idx[2]])
        if circ is None:
            continue
        cx, cy, r = circ
        inliers = np.abs(np.hypot(pts[:, 0] - cx, pts[:, 1] - cy) - r) < tol_px
        if best_inliers is None or inliers.sum() > best_inliers.sum():
            best_inliers, best_circle = inliers, circ
    if best_inliers is None or best_inliers.sum() < max(3, int(min_inlier_frac * n)):
        return None
    cx, cy, r = _fit_circle_kasa(pts[best_inliers])
    return cx, cy, r, np.abs(np.hypot(pts[:, 0] - cx, pts[:, 1] - cy) - r) < tol_px


def _punctum_centroids(mask: np.ndarray) -> np.ndarray:
    lbl = measure.label(mask)
    return np.array([[r.centroid[1], r.centroid[0]]
                     for r in measure.regionprops(lbl)], dtype=float).reshape(-1, 2)


def detect_npc_puncta(npc_crop_raw, nucleus_mask_crop, droplet_mask_crop,
                      cfg: GoldStandardConfig) -> np.ndarray:
    """
    NPC puncta in an annular zone straddling the nucleus edge.

    `npc_crop_raw` MUST be the RAW NPC plane — the threshold is mean + k*std of
    raw intensity over the droplet interior, so any contrast stretch upstream
    invalidates it. Excludes the nucleus interior (antibody-diffusion artefact)
    and the droplet wall (the v6-era error) by construction.
    """
    if nucleus_mask_crop.sum() == 0:
        return np.zeros_like(nucleus_mask_crop, dtype=bool)
    npc = np.asarray(npc_crop_raw, np.float32)
    selem = morphology.disk(cfg.px(cfg.npc_margin_um))
    zone = (morphology.binary_dilation(nucleus_mask_crop, selem)
            & ~morphology.binary_erosion(nucleus_mask_crop, selem))
    vals = npc[droplet_mask_crop]
    if vals.size == 0:
        return np.zeros_like(nucleus_mask_crop, dtype=bool)
    puncta = (npc > vals.mean() + cfg.npc_std_mult * vals.std()) & zone
    min_px = int(cfg.um2_to_px(cfg.gate_puncta_min_size_um2))
    if min_px > 0:
        puncta = morphology.remove_small_objects(puncta, min_size=min_px)
    return puncta


def gate_npc_shell(puncta: np.ndarray, cfg: GoldStandardConfig):
    cents = _punctum_centroids(puncta)
    info = {"n_puncta": len(cents), "shell_r_px": np.nan, "shell_inliers": 0}
    if len(cents) < cfg.gate_min_puncta:
        return False, info
    fit = fit_circle_ransac(cents, tol_px=cfg.um(cfg.gate_shell_tol_um),
                            n_iter=cfg.ransac_n_iter,
                            min_inlier_frac=cfg.gate_shell_min_inlier_frac)
    if fit is None:
        return False, info
    _, _, r, inliers = fit
    info["shell_r_px"], info["shell_inliers"] = float(r), int(inliers.sum())
    return bool(inliers.sum() >= max(3, int(cfg.gate_shell_min_inlier_frac * len(cents)))), info


def gate_membrane_coloc(puncta, mem_crop_raw, droplet_mask_crop, cfg: GoldStandardConfig):
    cents = _punctum_centroids(puncta)
    info = {"coloc_frac": 0.0}
    if len(cents) == 0:
        return False, info
    mem = np.asarray(mem_crop_raw, np.float32)
    interior = mem[droplet_mask_crop]
    if interior.size == 0:
        return False, info
    present = mem > interior.mean() + cfg.gate_membrane_k_std * interior.std()
    dil = morphology.binary_dilation(
        present, morphology.disk(cfg.px(cfg.gate_membrane_nn_radius_um)))
    H, W = mem.shape
    supported = sum(1 for x, y in cents
                    if 0 <= int(round(y)) < H and 0 <= int(round(x)) < W
                    and dil[int(round(y)), int(round(x))])
    info["coloc_frac"] = float(supported / len(cents))
    return info["coloc_frac"] >= cfg.gate_membrane_min_coloc_frac, info


def npc_ring_contrast(npc_crop_raw, nucleus_mask_crop, cfg: GoldStandardConfig) -> float:
    """Raw NPC in the envelope annulus / raw NPC in the eroded core.

    The measurement that actually separates an envelope from a flat cap
    (v16.2 measured 1.46 vs 1.00 on the synthetic check).
    """
    sel = morphology.disk(cfg.px(cfg.npc_margin_um))
    zone = (morphology.binary_dilation(nucleus_mask_crop, sel)
            & ~morphology.binary_erosion(nucleus_mask_crop, sel))
    core = morphology.binary_erosion(nucleus_mask_crop, sel)
    if not core.any():
        core = nucleus_mask_crop
    npc = np.asarray(npc_crop_raw, np.float32)
    if not zone.any() or not core.any():
        return float("nan")
    core_mean = float(npc[core].mean())
    return float(npc[zone].mean() / core_mean) if core_mean > 0 else float("nan")


def stage2_gate(nucleus_mask_crop, npc_crop_raw, mem_crop_raw, droplet_mask_crop,
                cfg: GoldStandardConfig, t_idx: Optional[int] = None):
    """
    Full Stage-2 verdict for one candidate. Returns (passed, info).

    Below `gate_min_timepoint` the gate abstains (`passed=None`) rather than
    failing: at t = 0-1 there are too few puncta to fit a shell and the NE probe
    is too dim to co-localise, so a failure there carries no information. §10
    writes abstentions as UNANNOTATED.
    """
    info = {"n_puncta": 0, "shell_r_px": np.nan, "shell_inliers": 0,
            "coloc_frac": np.nan, "ring_contrast": np.nan,
            "shell_ok": False, "membrane_ok": False, "ring_contrast_ok": False,
            "stage2": "fail"}
    if cfg.gate_min_timepoint is not None and t_idx is not None and t_idx < cfg.gate_min_timepoint:
        info["stage2"] = "abstain"
        return None, info
    if nucleus_mask_crop.sum() == 0:
        return False, info

    puncta = detect_npc_puncta(npc_crop_raw, nucleus_mask_crop, droplet_mask_crop, cfg)
    shell_ok, shell_info = gate_npc_shell(puncta, cfg)
    mem_ok, mem_info = gate_membrane_coloc(puncta, mem_crop_raw, droplet_mask_crop, cfg)
    ring = npc_ring_contrast(npc_crop_raw, nucleus_mask_crop, cfg)
    ring_ok = (cfg.gate_min_ring_contrast is None
               or (np.isfinite(ring) and ring >= cfg.gate_min_ring_contrast))

    info.update(shell_info); info.update(mem_info)
    info.update({"ring_contrast": float(ring), "shell_ok": bool(shell_ok),
                 "membrane_ok": bool(mem_ok), "ring_contrast_ok": bool(ring_ok)})
    passed = bool(shell_ok and mem_ok and ring_ok)
    info["stage2"] = "pass" if passed else "fail"
    return passed, info

### 6c. Legacy detector, kept for A/B only

`detect_nucleus_adaptive_legacy` is `Label_Generation_QC` §Stage 1 verbatim,
with two diagnostics added: `n_fragments` (connected components in the
thresholded mask **before** the largest-component pick) and
`largest_frac_of_thresholded` (how much of the thresholded signal the returned
mask actually kept). Those two numbers are the fragmentation symptom, measured.
It is never used to build labels.

In [ ]:
def detect_nucleus_adaptive_legacy(nls_crop, droplet_mask_crop, cfg: GoldStandardConfig):
    nls = np.asarray(nls_crop, np.float32)
    droplet_area = int(droplet_mask_crop.sum())
    empty = np.zeros_like(droplet_mask_crop, dtype=bool)
    info = {"method": "none", "n_fragments": 0, "largest_frac_of_thresholded": np.nan,
            "area_px": 0, "frac_of_droplet": np.nan}
    if droplet_area == 0:
        return empty, info

    diameter = 2.0 * np.sqrt(droplet_area / np.pi)
    bs = max(3, int(diameter / 3))
    bs = bs + 1 if bs % 2 == 0 else bs

    def _clean_and_gate(raw):
        mask = raw & droplet_mask_crop
        mask = morphology.remove_small_objects(mask, min_size=64)
        mask = ndi.binary_fill_holes(mask)
        n_frag = int(measure.label(mask).max())
        frac = mask.sum() / droplet_area
        if not (cfg.nuc_min_frac_of_droplet <= frac <= cfg.nuc_max_frac_of_droplet):
            return None, n_frag, mask
        lbl = measure.label(mask)
        props = measure.regionprops(lbl)
        if not props:
            return None, n_frag, mask
        big = max(props, key=lambda r: r.area)
        if _circularity(big) < 0.40:
            return None, n_frag, mask
        return (lbl == big.label), n_frag, mask

    for method, raw in (("adaptive", None), ("otsu", None)):
        try:
            if method == "adaptive":
                raw = nls > filters.threshold_local(nls, block_size=bs)
            else:
                vals = nls[droplet_mask_crop]
                if not (vals.size and vals.max() > vals.min()):
                    continue
                raw = nls > filters.threshold_otsu(vals)
        except Exception:
            continue
        cand, n_frag, pre = _clean_and_gate(raw)
        if cand is not None:
            info.update({"method": method, "n_fragments": n_frag,
                         "largest_frac_of_thresholded": float(cand.sum() / max(pre.sum(), 1)),
                         "area_px": int(cand.sum()),
                         "frac_of_droplet": float(cand.sum() / droplet_area)})
            return cand, info
        info["n_fragments"] = max(info["n_fragments"], n_frag)
    return empty, info

## 7. Synthetic self-test — run this before touching real data

A droplet phantom with a known nucleus and bright interior patches. Runs in
seconds, needs no image, and answers three questions the real data cannot
answer directly because it has no ground truth:

1. Does the new detector recover the **whole** interior? (Dice, and area
   relative to truth — Dice alone hides a mask that is the right shape and the
   wrong size.)
2. Does the legacy detector actually fail on this input, and how?
3. Does the watershed hallucinate a nucleus in an **empty** droplet, and do
   the gates catch it?

Two regimes, because the answer differs between them:

* **`uniform`** — bright nucleus (1.8× cytoplasm), mild interior patches
  (1.6× nuclear). Resembles a late timepoint with well-accumulated NLS.
* **`heterogeneous`** — dim nucleus (1.3× cytoplasm), strong interior patches
  (2.6× nuclear). This is the reported failure: the patches, not the nucleus,
  dominate the intensity histogram.

Measured on this machine at `pixel_size_um = 0.1625`, 8 droplets per regime:

| Regime | legacy Dice | legacy area/truth | new Dice | new area/truth |
|---|---|---|---|---|
| uniform | 0.96 | 0.93 | 0.99 | 0.98 |
| heterogeneous | 0.49 | **0.34** | 0.98 | 0.98 |

Read the `area/truth` column, not Dice: in the heterogeneous regime the legacy
detector returns **a third of the nucleus**. Every downstream measurement
inherits that — area directly, N/C ratio because the surviving mask sits on
the brightest interior pixels, radial sweeps because the ray origin and the
normalising surface both move.

Empty-droplet control: the watershed floods to 0.78 of droplet area in every
trial, as it must, and the area-fraction gate rejects it in every trial. The
assertion at the end of the cell is the one that matters — if it ever fails,
the gold standard is being populated with hallucinated nuclei.

In [ ]:
PHANTOM_REGIMES = {
    # nucleus brightness relative to cytoplasm, patch brightness relative to
    # nucleus, number of patches
    "uniform":       dict(nuc=1.8, patch_gain=1.6, n_patch=7),
    "heterogeneous": dict(nuc=1.3, patch_gain=2.6, n_patch=10),
}


def make_droplet_phantom(cfg: GoldStandardConfig, seed: int = 0,
                         regime: str = "uniform",
                         r_drop_um: float = 12.0, r_nuc_um: float = 6.0,
                         cyto: float = 1.0, noise: float = 0.03,
                         with_nucleus: bool = True, **overrides):
    """Droplet + nucleus + locally bright interior patches. Returns (nls, droplet, truth)."""
    params = dict(PHANTOM_REGIMES[regime]); params.update(overrides)
    nuc, patch_gain, n_patch = params["nuc"], params["patch_gain"], params["n_patch"]
    px = cfg.pixel_size_um
    prng = np.random.default_rng(seed)
    R = int(r_drop_um / px) + 14
    yy, xx = np.mgrid[-R:R + 1, -R:R + 1].astype(np.float32)
    rr = np.hypot(yy, xx) * px
    droplet = rr <= r_drop_um
    truth = rr <= r_nuc_um
    img = np.where(droplet, cyto, 0.15).astype(np.float32)
    if with_nucleus:
        img[truth] = nuc
        for _ in range(n_patch):
            a = prng.uniform(0, 2 * np.pi)
            d = prng.uniform(0, r_nuc_um * 0.65)
            cy, cx = d * np.sin(a) / px, d * np.cos(a) / px
            patch = np.hypot(yy - cy, xx - cx) * px <= prng.uniform(1.0, 2.0)
            img[patch & truth] = nuc * patch_gain
    img = filters.gaussian(img, sigma=0.4 / px, preserve_range=True)
    img = img + prng.normal(0, noise, img.shape)
    return img.astype(np.float32), droplet, truth


def dice(a: np.ndarray, b: np.ndarray) -> float:
    s = int(a.sum()) + int(b.sum())
    return float(2.0 * np.logical_and(a, b).sum() / s) if s else float("nan")


def run_synthetic_ab(cfg: GoldStandardConfig, n: int = 8,
                     regimes: Sequence[str] = ("uniform", "heterogeneous")
                     ) -> pd.DataFrame:
    rows = []
    for regime in regimes:
        for s in range(n):
            nls, drop, truth = make_droplet_phantom(cfg, seed=s, regime=regime)
            m_leg, i_leg = detect_nucleus_adaptive_legacy(nls, drop, cfg)
            m_ws, i_ws = segment_nucleus_watershed(nls, drop, cfg)
            rows.append({
                "regime": regime, "seed": s,
                "legacy_dice": dice(m_leg, truth), "legacy_method": i_leg["method"],
                "legacy_fragments": i_leg["n_fragments"],
                "legacy_area_ratio": (m_leg.sum() / truth.sum()) if truth.any() else np.nan,
                "ws_dice": dice(m_ws, truth), "ws_seeds_raw": i_ws["n_seeds_raw"],
                "ws_area_ratio": (m_ws.sum() / truth.sum()) if truth.any() else np.nan,
                "ws_accepted": i_ws["accepted"], "ws_reject": i_ws["reject"],
            })
    return pd.DataFrame(rows)


def run_synthetic_controls(cfg: GoldStandardConfig, n: int = 4) -> pd.DataFrame:
    """Empty droplets: the watershed MUST flood, and the gates MUST reject."""
    rows = []
    for s in range(n):
        nls, drop, _ = make_droplet_phantom(cfg, seed=500 + s, with_nucleus=False)
        m_ws, i_ws = segment_nucleus_watershed(nls, drop, cfg)
        rows.append({"seed": s, "flooded_frac": float(m_ws.sum() / drop.sum()),
                     "accepted": i_ws["accepted"], "reject": i_ws["reject"],
                     "ne_contrast": i_ws["ne_contrast"]})
    return pd.DataFrame(rows)


ab = run_synthetic_ab(cfg)
summary = (ab.groupby("regime")[["legacy_dice", "legacy_area_ratio",
                                 "ws_dice", "ws_area_ratio"]]
             .mean().round(3)
             .rename(columns={"legacy_area_ratio": "legacy_area/truth",
                              "ws_area_ratio": "ws_area/truth"}))
print("A/B on synthetic droplets  (area/truth = 1.0 means correct extent)")
print(summary.to_string())
print("\nper-droplet detail:")
print(ab.round(3).to_string(index=False))

ctrl = run_synthetic_controls(cfg)
print("\nempty-droplet control (accepted must be False everywhere)")
print(ctrl.round(3).to_string(index=False))

assert not ctrl.accepted.any(), "GATE FAILURE: a nucleus was accepted in an empty droplet"
het = ab[ab.regime == "heterogeneous"]
assert het.ws_area_ratio.mean() > 1.5 * het.legacy_area_ratio.mean(), (
    "the new detector no longer recovers materially more of the nucleus than the "
    "legacy one on the heterogeneous regime — re-check nuc_flatten_um")
print("\nself-test passed.")

### 7b. Visual check and the `nuc_flatten_um` sweep

The left panel is the input; the second is what the reconstruction flattening
leaves; the third is the gradient the watershed floods. If flattening is doing
its job, the brightest ridge in panel 3 is the envelope, not the patch edges.

The sweep is the honest version of "3.0 µm is the default": the parameter has
a working range, it has an upper limit set by nuclear radius, and both are
visible in the numbers rather than asserted.

In [ ]:
def plot_flatten_stages(cfg: GoldStandardConfig, seed: int = 0,
                        regime: str = "heterogeneous"):
    nls, drop, truth = make_droplet_phantom(cfg, seed=seed, regime=regime)
    lo, hi = np.percentile(nls[drop], [1, 99.5])
    norm = np.clip((nls - lo) / (hi - lo), 0, 1)
    sm = filters.gaussian(norm, sigma=max(cfg.um(cfg.nuc_smooth_um), 0.5),
                          preserve_range=True).astype(np.float32)
    flat = flatten_interior(sm, cfg.px(cfg.nuc_flatten_um))
    grad = filters.sobel(flat)
    mask, info = segment_nucleus_watershed(nls, drop, cfg)

    fig, ax = plt.subplots(1, 4, figsize=(17, 4.4))
    for a, im, ti in zip(ax, [sm, flat, grad, sm],
                         ["smoothed NLS", f"flattened (SE r={info['flatten_se_px']}px)",
                          "gradient flooded by watershed", "result"]):
        a.imshow(im, cmap="magma"); a.set_title(ti, fontsize=10); a.axis("off")
    ax[3].contour(truth, levels=[0.5], colors="w", linewidths=1.2)
    ax[3].contour(mask, levels=[0.5], colors="c", linewidths=1.2)
    ax[3].set_title(f"white = truth, cyan = result (Dice {dice(mask, truth):.3f})",
                    fontsize=10)
    plt.tight_layout(); plt.show()


def sweep_flatten_radius(cfg: GoldStandardConfig,
                         radii_um=(2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0),
                         r_nuc_um=(3.0, 4.5, 6.0, 9.0),
                         regime: str = "heterogeneous") -> pd.DataFrame:
    """
    The clamp is forced OFF here. With it on, every requested radius above
    ~0.35 x droplet radius silently collapses to the same value, so rows of the
    sweep come out identical and look like a plateau that does not exist.
    """
    from dataclasses import replace
    rows = []
    for r_um in radii_um:
        c2 = replace(cfg, nuc_flatten_um=r_um, nuc_flatten_max_r_frac=None)
        for rn in r_nuc_um:
            ds = []
            for s in range(6):
                nls, drop, truth = make_droplet_phantom(c2, seed=s, regime=regime,
                                                        r_nuc_um=rn)
                m, _ = segment_nucleus_watershed(nls, drop, c2)
                ds.append(dice(m, truth))
            rows.append({"flatten_um": r_um, "r_nucleus_um": rn,
                         "mean_dice": np.mean(ds), "min_dice": np.min(ds)})
    return pd.DataFrame(rows)


plot_flatten_stages(cfg, regime="heterogeneous")
for regime in ("heterogeneous", "uniform"):
    sweep = sweep_flatten_radius(cfg, regime=regime)
    print(f"\nmean Dice — regime = {regime}")
    print(sweep.pivot(index="flatten_um", columns="r_nucleus_um",
                      values="mean_dice").round(3).to_string())
print("\nRows = SE radius (µm), columns = true nuclear radius (µm).")
print("Too small an SE leaves interior patches standing; too large erodes the")
print("nucleus itself. The safe band narrows as the nucleus gets smaller —")
print(f"current setting is nuc_flatten_um = {cfg.nuc_flatten_um} µm.")

### 7c. Guard test — nuclear area must not depend on droplet size

This project measures how nuclear size responds to droplet size. Any
segmentation parameter that scales with droplet radius therefore writes the
conclusion into the method.

`nuc_flatten_max_r_frac` did exactly that. With the clamp at 0.35 × droplet
radius, the *same* 113 µm² nucleus measured:

| droplet | 201 µm² | 340 µm² | 499 µm² | 804 µm² | 1507 µm² |
|---|---|---|---|---|---|
| SE used | 2.76 µm | 3.58 µm | 4.06 µm | 4.06 µm | 4.06 µm |
| area bias | **−54 %** | −10 % | −2 % | −2 % | −2 % |

— a spurious positive nuclear-size/droplet-size correlation, strongest exactly
where droplets are smallest. With the clamp off the bias is a flat −2 % at
every droplet size. The clamp is now disabled by default and this test is here
so it stays that way: it fails if any droplet-size-dependent term is
reintroduced into the segmentation.

In [ ]:
def guard_no_droplet_size_bias(cfg: GoldStandardConfig,
                               droplet_r_um=(8.0, 10.4, 12.6, 16.0, 21.9),
                               r_nuc_um: float = 6.0, n: int = 6,
                               tol_pp: float = 8.0) -> pd.DataFrame:
    """Same nucleus, different droplets. Measured area must not move with droplet size."""
    true_um2 = np.pi * r_nuc_um ** 2
    rows = []
    for r_drop in droplet_r_um:
        areas, se_used = [], np.nan
        for s in range(n):
            nls, drop, _ = make_droplet_phantom(cfg, seed=s, regime="heterogeneous",
                                                r_drop_um=r_drop, r_nuc_um=r_nuc_um)
            m, i = segment_nucleus_watershed(nls, drop, cfg)
            areas.append(cfg.px_to_um2(int(m.sum())))
            se_used = i.get("flatten_se_px", np.nan) * cfg.pixel_size_um
        rows.append({"droplet_um2": np.pi * r_drop ** 2, "se_used_um": se_used,
                     "measured_um2": float(np.mean(areas)), "true_um2": true_um2,
                     "bias_pct": 100 * (np.mean(areas) - true_um2) / true_um2})
    df = pd.DataFrame(rows)
    spread = df.bias_pct.max() - df.bias_pct.min()
    df.attrs["spread_pp"] = spread
    assert spread < tol_pp, (
        f"DROPLET-SIZE BIAS: measured area of a fixed nucleus varies by "
        f"{spread:.1f} percentage points across droplet sizes. Some segmentation "
        f"parameter is scaling with droplet radius — this manufactures the "
        f"nuclear-scaling correlation being measured.")
    return df


guard = guard_no_droplet_size_bias(cfg)
print(guard.round(2).to_string(index=False))
print(f"\nbias spread across droplet sizes: {guard.attrs['spread_pp']:.1f} "
      f"percentage points — flat means no droplet-size dependence.")

## 8. Real-data audit — does the fragmentation story hold on the FOV?

The synthetic test proves the *mechanism*. This measures whether the mechanism
is what is actually happening in `control_extract_1.1.tif`, on real droplets,
before any of it is used to build a gold standard.

Per droplet, both detectors are run and the diagnostic is recorded. What to
look for:

| Column | Fragmentation signature |
|---|---|
| `legacy_fragments` | > 1 → the threshold split one nucleus |
| `legacy_largest_frac` | ≪ 1 → the returned mask is a fragment of what was thresholded |
| `area_ratio_ws_over_legacy` | ≫ 1 → the legacy mask was under-extended |
| `ws_ne_contrast` | should stay > 1.05 while area grows — proves the extra area is real nucleus, not flood |

That last column is the control on the claim. Growing the mask always
increases area; it only counts as a fix if the added pixels are still
NLS-enriched relative to the surrounding cytoplasm.

In [ ]:
def audit_plane(hyperstack, t: int, z: int, cfg: GoldStandardConfig,
                max_droplets: Optional[int] = None) -> pd.DataFrame:
    """Run both nucleus detectors on every droplet in one plane and tabulate."""
    npc_clip_src = extract_plane(hyperstack, t, z, cfg.ch_npc)
    droplets = detect_droplets(npc_clip_src, cfg)
    if max_droplets:
        droplets = droplets[:max_droplets]
    if not droplets:
        return pd.DataFrame()

    nls = extract_plane(hyperstack, t, z, cfg.ch_nls)
    npc_raw = extract_plane(hyperstack, t, z, cfg.ch_npc)      # re-pulled RAW
    mem_raw = extract_plane(hyperstack, t, z, cfg.ch_membrane)

    rows = []
    for i, d in enumerate(droplets):
        bb = d["bbox"]
        dm = crop_bbox(d["mask"], bb)
        nls_c, npc_c, mem_c = (crop_bbox(a, bb) for a in (nls, npc_raw, mem_raw))

        m_leg, i_leg = detect_nucleus_adaptive_legacy(nls_c, dm, cfg)
        m_ws, i_ws = segment_nucleus_watershed(nls_c, dm, cfg)
        s2_pass, s2 = stage2_gate(m_ws, npc_c, mem_c, dm, cfg, t_idx=t)

        a_leg, a_ws = int(m_leg.sum()), int(m_ws.sum())
        rows.append({
            "t": t, "z": z, "droplet_idx": i,
            "droplet_area_um2": d["area_um2"], "droplet_circ": d["circ"],
            "centroid_y": d["centroid"][0], "centroid_x": d["centroid"][1],
            "legacy_method": i_leg["method"], "legacy_fragments": i_leg["n_fragments"],
            "legacy_largest_frac": i_leg["largest_frac_of_thresholded"],
            "legacy_area_um2": cfg.px_to_um2(a_leg),
            "ws_area_um2": cfg.px_to_um2(a_ws),
            "area_ratio_ws_over_legacy": (a_ws / a_leg) if a_leg else np.nan,
            "iou_legacy_ws": (np.logical_and(m_leg, m_ws).sum()
                              / max(np.logical_or(m_leg, m_ws).sum(), 1)),
            "ws_frac_of_droplet": i_ws["frac_of_droplet"],
            "ws_solidity": i_ws["solidity"], "ws_ne_contrast": i_ws["ne_contrast"],
            "ws_accepted": i_ws["accepted"], "ws_reject": i_ws["reject"],
            "stage2": s2["stage2"], "n_puncta": s2["n_puncta"],
            "ring_contrast": s2["ring_contrast"], "coloc_frac": s2["coloc_frac"],
        })
    return pd.DataFrame(rows)


def summarise_audit(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    g = df.groupby("t")
    return pd.DataFrame({
        "droplets": g.size(),
        "legacy_fragmented_%": g.apply(lambda d: 100 * (d.legacy_fragments > 1).mean()),
        "legacy_none_%": g.apply(lambda d: 100 * (d.legacy_method == "none").mean()),
        "median_legacy_largest_frac": g.legacy_largest_frac.median(),
        "median_area_ratio_ws/legacy": g.area_ratio_ws_over_legacy.median(),
        "median_ws_area_um2": g.ws_area_um2.median(),
        "ws_accepted_%": g.apply(lambda d: 100 * d.ws_accepted.mean()),
        "median_ws_ne_contrast": g.ws_ne_contrast.median(),
        "stage2_pass_%": g.apply(lambda d: 100 * (d.stage2 == "pass").mean()),
    }).round(2)


def plot_audit_gallery(hyperstack, t: int, z: int, cfg: GoldStandardConfig,
                       n: int = 6, pad_um: float = 2.0):
    """Side-by-side crops: raw NLS, legacy mask, watershed mask, per-droplet."""
    droplets = detect_droplets(extract_plane(hyperstack, t, z, cfg.ch_npc), cfg)[:n]
    if not droplets:
        print("no droplets"); return
    nls = extract_plane(hyperstack, t, z, cfg.ch_nls)
    pad = cfg.px(pad_um)
    fig, ax = plt.subplots(len(droplets), 3, figsize=(9, 3 * len(droplets)))
    ax = np.atleast_2d(ax)
    for r, d in enumerate(droplets):
        r0, c0, r1, c1 = d["bbox"]
        r0, c0 = max(r0 - pad, 0), max(c0 - pad, 0)
        r1, c1 = min(r1 + pad, nls.shape[0]), min(c1 + pad, nls.shape[1])
        bb = (r0, c0, r1, c1)
        dm, nls_c = crop_bbox(d["mask"], bb), crop_bbox(nls, bb)
        m_leg, i_leg = detect_nucleus_adaptive_legacy(nls_c, dm, cfg)
        m_ws, i_ws = segment_nucleus_watershed(nls_c, dm, cfg)
        lo, hi = np.percentile(nls_c, [1, 99])
        disp = np.clip((nls_c - lo) / (hi - lo + 1e-8), 0, 1)
        for c, (im, ti) in enumerate([
                (disp, "raw NLS"),
                (m_leg, f"legacy {i_leg['method']} · {i_leg['n_fragments']} frag"),
                (m_ws, f"watershed · {'OK' if i_ws['accepted'] else i_ws['reject'][:22]}")]):
            ax[r, c].imshow(disp, cmap="gray")
            if c:
                ax[r, c].contour(im, levels=[0.5], colors="c", linewidths=1.1)
            ax[r, c].set_title(ti, fontsize=8); ax[r, c].axis("off")
    plt.tight_layout(); plt.show()

### 8b. Run the audit

Set `RUN_AUDIT = True` once the hyperstack is reachable. Sample a few
timepoints spanning the NPC progression (early puncta-only → complete ring),
because the two detectors are expected to disagree *most* at late timepoints
where NLS is bright and interior structure is strongest.

In [ ]:
RUN_AUDIT = False          # ← flip to True on Cheaha

if RUN_AUDIT:
    hyperstack = load_hyperstack(gs_paths.image_path)
    T, Z = hyperstack.shape[0], hyperstack.shape[1]
    assert_histogram_isolation(hyperstack, t=min(4, T - 1), z=min(8, Z - 1))

    audit_t = [t for t in (2, 4, 6, 8, 9) if t < T]
    audit_z = min(16, Z - 1)
    audit = pd.concat(
        [audit_plane(hyperstack, t, audit_z, cfg, max_droplets=25) for t in audit_t],
        ignore_index=True)
    audit.to_csv(gs_paths.qc_dir / "detector_ab_audit.csv", index=False)
    display(summarise_audit(audit))
    plot_audit_gallery(hyperstack, t=audit_t[-1], z=audit_z, cfg=cfg, n=6)
else:
    print("RUN_AUDIT is False — skipping. Set it True once the image is reachable.")

## 8c. Abnormal-nucleus scanner

Flag nuclei whose size is anomalous for their timepoint, then look at them.
Three scanners, because one is not enough and the reason why is the whole
design of this section.

### Why raw area vs. the timepoint mean does not work here

Nuclear size depends on droplet size — that is the phenomenon under study.
Droplets in this dataset span 200–1500 µm², so a perfectly segmented nucleus
in a small droplet looks "abnormally small" against the timepoint's central
value. Measured on a synthetic population with four fragmented and two merged
nuclei injected:

| Droplet range | scan on raw area | scan on droplet-conditioned residual |
|---|---|---|
| wide (200–1400 µm²) | **0 / 5 found** | 5 / 5 found, 0 false alarms |
| narrow (~600 µm²) | 5 / 5 found | 5 / 5 found |

On a realistic droplet range the raw-area scan finds **nothing**: genuine
size variation swamps the segmentation failures. `scan_size_outliers`
therefore fits nucleus area against droplet area per timepoint (Theil-Sen, in
log-log, robust to the outliers being searched for) and scores the residual.

### Mean, median or mode

Mean is out — a handful of fragments drags it down and lowers the very
threshold meant to catch them. Median + MAD is the default and behaved best:
0 false alarms at every contamination level tested.

Mode was worth testing and the result is mixed, so both are reported and
neither is trusted blindly. A half-sample mode recovered the clean centre
better than the median at 33 % contamination (22.2 vs 13.3 µm² against a true
21.0), but as a *detector* it was erratic — 2/12 found at 20 % contamination,
then 27/27 with 12 false alarms at 45 %. It is reported as a statistic, not
used as the threshold.

### The failure mode you must know about

**A near-zero flag rate is ambiguous, not reassuring.** Recall against
contamination rate, median + MAD on residuals, n = 60:

| fragmented fraction | 5 % | 10 % | 20 % | 33 % | 45 % |
|---|---|---|---|---|---|
| found | 3/3 | 5/6 | 8/12 | **0/19** | **0/27** |
| % of population flagged | 5.0 | 8.3 | 13.3 | **0.0** | **0.0** |

Past ~20 % the scanner flags *nothing* — the failures have become
the population, and there is no outlier left to be an outlier against. A quiet
scanner means either "segmentation is clean" or "segmentation is uniformly
broken", and the flag count alone cannot tell you which.

That is why the other two scanners exist, and both use a reference the
population cannot contaminate:

* `scan_absolute_prior` — nuclear area against a stated physical range
  (100–600 µm² for this dataset). Immune to contamination because the
  reference is external: at 45 % contamination it recovered **17 of 27**
  injected failures where the population scan recovered 0. It misses the
  milder ones — a fragment of a large nucleus can still land inside the
  physical range — so it is a floor, not a replacement.
* `scan_z_consistency` — the same nucleus segmented on adjacent z-planes.
  Self-controlled: a plane that fragments shows an area dip against its own
  neighbours, however many other nuclei are also broken.

Run all three. Agreement is informative; `scan_size_outliers` alone is not.

In [ ]:
MAD_TO_SIGMA = 1.4826


def half_sample_mode(x: np.ndarray, min_n: int = 4) -> float:
    """Bickel half-sample mode — recursively keep the densest half. No binning."""
    x = np.sort(np.asarray(x, float))
    x = x[np.isfinite(x)]
    n = len(x)
    if n == 0:
        return float("nan")
    while n > min_n:
        h = n // 2
        i = int(np.argmin(x[h:] - x[:n - h]))
        x = x[i:i + h + 1]
        n = len(x)
    return float(np.mean(x))


def robust_z(values: np.ndarray) -> Tuple[np.ndarray, float, float]:
    """(z, centre, scale) from median + MAD, falling back to IQR then std."""
    v = np.asarray(values, float)
    med = float(np.median(v))
    scale = float(np.median(np.abs(v - med)) * MAD_TO_SIGMA)
    if scale <= 0:
        iqr = float(np.subtract(*np.percentile(v, [75, 25])))
        scale = iqr / 1.349 if iqr > 0 else float(np.std(v))
    if scale <= 0:
        return np.zeros_like(v), med, float("nan")
    return (v - med) / scale, med, scale


def scan_size_outliers(metrics: pd.DataFrame, cfg: GoldStandardConfig,
                       z_thresh: float = 3.5, min_n_for_fit: int = 8,
                       condition_on_droplet: bool = True) -> pd.DataFrame:
    """
    Flag size-anomalous nuclei per timepoint.

    Needs columns: t, z, droplet_area_um2, nucleus_area_um2 (+ droplet_id if
    present). Optional columns (solidity, ne_contrast, frac_of_droplet,
    ring_contrast, n_puncta) are carried through and used for the diagnosis hint.

    Adds: resid, z_size, flag ('small'/'large'/''), diagnosis, and per-timepoint
    context columns (centre, scale, slope, n_used, pct_flagged, mode_um2).
    """
    from scipy import stats as _stats

    need = {"t", "droplet_area_um2", "nucleus_area_um2"}
    missing = need - set(metrics.columns)
    if missing:
        raise ValueError(f"metrics is missing {sorted(missing)}")

    out = []
    for t, g in metrics.groupby("t", sort=True):
        g = g.copy()
        ok = (g.nucleus_area_um2 > 0) & np.isfinite(g.nucleus_area_um2)
        g["log_area"] = np.log(g.nucleus_area_um2.where(ok))

        slope = np.nan
        if condition_on_droplet and int(ok.sum()) >= min_n_for_fit:
            x = np.log(g.droplet_area_um2[ok].to_numpy())
            y = g.log_area[ok].to_numpy()
            slope, intercept, _, _ = _stats.theilslopes(y, x)
            g["resid"] = g.log_area - (intercept + slope * np.log(g.droplet_area_um2))
        else:
            g["resid"] = g.log_area          # too few nuclei to fit — raw fallback

        if int(ok.sum()) >= 3:
            z, centre, scale = robust_z(g.resid[ok].to_numpy())
            g.loc[ok, "z_size"] = z
            # mode of the raw log-area, reported as a statistic only — see 8c
            g["mode_um2"] = float(np.exp(half_sample_mode(g.log_area[ok].to_numpy())))
        else:
            g["z_size"], centre, scale = np.nan, np.nan, np.nan
            g["mode_um2"] = np.nan

        g["centre_resid"] = centre
        g["scale_resid"] = scale
        g["droplet_slope"] = slope
        g["n_used"] = int(ok.sum())
        out.append(g)

    r = pd.concat(out, ignore_index=True)
    r["flag"] = np.where(r.z_size <= -z_thresh, "small",
                np.where(r.z_size >= z_thresh, "large", ""))

    # per-timepoint flag rate — read this alongside the flags themselves
    rate = r.groupby("t").flag.apply(lambda s: 100.0 * (s != "").mean())
    r["pct_flagged_this_t"] = r.t.map(rate)
    r["diagnosis"] = [_diagnose(row) for _, row in r.iterrows()]
    return r.sort_values("z_size")


def _diagnose(row) -> str:
    """Map a flag plus the secondary features onto a likely cause."""
    if row.get("flag") == "small":
        if row.get("solidity", 1.0) < 0.85:
            return "fragment or partial mask (low solidity)"
        if row.get("ne_contrast", 9.9) < 1.15:
            return "weak NLS enrichment — possible false positive"
        return "under-extended mask — check nuc_flatten_um"
    if row.get("flag") == "large":
        frac = row.get("frac_of_droplet", np.nan)
        if np.isfinite(frac) and frac > 0.40:
            return "wall flood or whole-droplet false positive"
        if row.get("solidity", 1.0) < 0.85:
            return "merged nuclei (concave outline)"
        return "over-extended mask or genuinely large nucleus"
    return ""


def scan_absolute_prior(metrics: pd.DataFrame, cfg: GoldStandardConfig,
                        min_um2: float = 100.0, max_um2: float = 600.0
                        ) -> pd.DataFrame:
    """
    Flag against a stated physical range rather than the population.

    Contamination-immune: the reference is external, so this still works when
    most nuclei are broken and scan_size_outliers has gone quiet. The bounds are
    the same ones filter_nuclei_by_area uses in the pipeline — change them here
    and there together.
    """
    r = metrics.copy()
    a = r.nucleus_area_um2
    r["abs_flag"] = np.where(a <= 0, "empty",
                    np.where(a < min_um2, "below_prior",
                    np.where(a > max_um2, "above_prior", "")))
    r["abs_ratio"] = np.where(a < min_um2, a / min_um2,
                     np.where(a > max_um2, a / max_um2, 1.0))
    return r


def scan_z_consistency(hyperstack, t: int, z_centre: int, cfg: GoldStandardConfig,
                       dz: int = 1, max_droplets: Optional[int] = None,
                       drop_tol: float = 0.35) -> pd.DataFrame:
    """
    Self-controlled check: segment each droplet at z-dz, z, z+dz and compare.

    A nucleus that fragments on one plane shows an area dip against its own
    neighbouring planes. Because the reference is the same object, this holds
    however many *other* nuclei are also broken — the contamination limit that
    breaks scan_size_outliers does not apply.

    `area_dip` = 1 - area(z) / mean(area(z-dz), area(z+dz)). Flagged when it
    exceeds drop_tol, i.e. the centre plane lost more than that fraction.
    """
    Z = hyperstack.shape[1]
    zs = [z for z in (z_centre - dz, z_centre, z_centre + dz) if 0 <= z < Z]
    if z_centre not in zs or len(zs) < 3:
        raise ValueError(f"need z-{dz}, z, z+{dz} inside [0, {Z-1}]")

    droplets = detect_droplets(extract_plane(hyperstack, t, z_centre, cfg.ch_npc), cfg)
    if max_droplets:
        droplets = droplets[:max_droplets]

    planes = {z: extract_plane(hyperstack, t, z, cfg.ch_nls) for z in zs}
    rows = []
    for i, d in enumerate(droplets, start=1):
        bb = d["bbox"]
        dm = crop_bbox(d["mask"], bb)
        areas = {}
        for z in zs:
            m, _ = segment_nucleus_watershed(crop_bbox(planes[z], bb), dm, cfg)
            areas[z] = cfg.px_to_um2(int(m.sum()))
        nb = [areas[z] for z in zs if z != z_centre]
        nb_mean = float(np.mean(nb)) if nb else np.nan
        dip = 1.0 - areas[z_centre] / nb_mean if nb_mean > 0 else np.nan
        rows.append({
            "t": t, "z": z_centre, "droplet_id": i,
            "centroid_y": d["centroid"][0], "centroid_x": d["centroid"][1],
            "droplet_area_um2": d["area_um2"],
            **{f"area_z{z - z_centre:+d}_um2": areas[z] for z in zs},
            "neighbour_mean_um2": nb_mean, "area_dip": dip,
            "z_flag": "dip" if (np.isfinite(dip) and dip > drop_tol) else
                      ("spike" if (np.isfinite(dip) and dip < -drop_tol) else ""),
        })
    return pd.DataFrame(rows).sort_values("area_dip", ascending=False)

### 8d. Gallery — look at what was flagged

A flag is a hypothesis. The gallery reads back the saved proposal `.npz`
(image plus droplet and nucleus instance maps) and crops each flagged droplet,
so what you see is exactly the mask that produced the number — not a
re-segmentation that might differ.

Sorted by severity, worst first. Print the panel and check whether the
`diagnosis` column matches what the crop actually shows; when it does not,
the diagnosis rules in `_diagnose` are the thing to fix, not the segmenter.

In [ ]:
def plot_outlier_gallery(flagged: pd.DataFrame, gs_paths: GoldStandardPaths,
                         cfg: GoldStandardConfig, n: int = 12,
                         pad_um: float = 3.0, source: str = "proposals",
                         flag_col: str = "flag") -> None:
    """
    flagged needs: t, z, droplet_id, plus the flag column. Reads
    <source>/t##_z##.npz — 'proposals' before review, 'committed' after.
    """
    sel = flagged[flagged[flag_col].astype(str) != ""].copy()
    if sel.empty:
        print("nothing flagged."); return
    if "z_size" in sel.columns:
        sel = sel.reindex(sel.z_size.abs().sort_values(ascending=False).index)
    sel = sel.head(n)

    src_dir = getattr(gs_paths, f"{source}_dir")
    pad = cfg.px(pad_um)
    cache: Dict[str, dict] = {}

    ncol = 3
    nrow = int(np.ceil(len(sel) / ncol))
    fig, ax = plt.subplots(nrow, ncol, figsize=(4.0 * ncol, 4.0 * nrow))
    ax = np.atleast_1d(ax).ravel()

    for k, (_, row) in enumerate(sel.iterrows()):
        sid = f"t{int(row.t):02d}_z{int(row.z):02d}"
        if sid not in cache:
            p = src_dir / f"{sid}.npz"
            if not p.exists():
                ax[k].set_title(f"{sid}: no {source} file", fontsize=8)
                ax[k].axis("off"); continue
            d = np.load(p, allow_pickle=False)
            cache[sid] = {"image": d["image"], "nuc": d["nucleus_instances"],
                          "drop": d["droplet_instances"]}
        c = cache[sid]
        did = int(row.droplet_id)
        ys, xs = np.where(c["drop"] == did)
        if ys.size == 0:
            ax[k].set_title(f"{sid} d{did}: not found", fontsize=8)
            ax[k].axis("off"); continue
        r0, c0 = max(ys.min() - pad, 0), max(xs.min() - pad, 0)
        r1 = min(ys.max() + pad, c["nuc"].shape[0])
        c1 = min(xs.max() + pad, c["nuc"].shape[1])

        nls = c["image"][1, r0:r1, c0:c1].astype(np.float32)
        lo, hi = np.percentile(nls, [1, 99])
        ax[k].imshow(np.clip((nls - lo) / (hi - lo + 1e-8), 0, 1), cmap="gray")
        ax[k].contour(c["drop"][r0:r1, c0:c1] == did, levels=[0.5],
                      colors="#ffd700", linewidths=0.8)
        ax[k].contour(c["nuc"][r0:r1, c0:c1] == did, levels=[0.5],
                      colors="c", linewidths=1.2)

        bits = [f"{sid} d{did}  {row[flag_col]}"]
        if "z_size" in row and np.isfinite(row.z_size):
            bits.append(f"z={row.z_size:+.1f}  {row.nucleus_area_um2:.0f}µm²")
        if "diagnosis" in row and row.diagnosis:
            bits.append(str(row.diagnosis))
        ax[k].set_title("\n".join(bits), fontsize=7.5)
        ax[k].axis("off")

    for a in ax[len(sel):]:
        a.axis("off")
    plt.tight_layout(); plt.show()


def outlier_report(flagged: pd.DataFrame) -> pd.DataFrame:
    """Per-timepoint summary. Read pct_flagged FIRST — see the note in 8c."""
    if flagged.empty:
        return flagged
    g = flagged.groupby("t")
    rep = pd.DataFrame({
        "n_nuclei": g.n_used.first(),
        "median_um2": g.nucleus_area_um2.median(),
        "mode_um2": g.mode_um2.first(),
        "droplet_slope": g.droplet_slope.first(),
        "n_small": g.flag.apply(lambda s: int((s == "small").sum())),
        "n_large": g.flag.apply(lambda s: int((s == "large").sum())),
        "pct_flagged": g.pct_flagged_this_t.first(),
    }).round(3)
    rep["read_as"] = np.where(
        rep.pct_flagged == 0, "AMBIGUOUS: clean, or uniformly broken — cross-check",
        np.where(rep.pct_flagged > 20, "too many to be outliers — suspect systematic failure",
                 "usable"))
    return rep

### 8e. Self-test of the scanner

Inject known fragmented and merged nuclei into a synthetic population and
check the scanner finds them. Also re-derives the contamination table quoted
in §8c, so the stated limit is checked rather than asserted.

In [ ]:
def make_synthetic_population(n: int = 60, seed: int = 0,
                              frag_frac: float = 0.07, merge_frac: float = 0.03,
                              droplet_spread: bool = True,
                              scaling_exp: float = 0.66) -> pd.DataFrame:
    """Nuclear area scales with droplet area (real biology) plus lognormal noise.
    Fragmented nuclei get 0.35x area; merged get 2.2x."""
    prng = np.random.default_rng(seed)
    d = (np.exp(prng.uniform(np.log(200), np.log(1400), n)) if droplet_spread
         else np.exp(prng.normal(np.log(600), 0.12, n)))
    # coefficient set so a 600 um2 droplet gives a ~250 um2 nucleus, matching
    # the real 100-600 um2 range the absolute-prior scan is calibrated against
    a = 3.60 * d ** scaling_exp * np.exp(prng.normal(0, 0.16, n))
    truth = np.array([""] * n, dtype=object)
    idx = prng.permutation(n)
    nf, nm = int(n * frag_frac), int(n * merge_frac)
    for i in idx[:nf]:
        a[i] *= 0.35; truth[i] = "small"
    for i in idx[nf:nf + nm]:
        a[i] *= 2.2; truth[i] = "large"
    return pd.DataFrame({"t": 5, "z": 16, "droplet_id": np.arange(1, n + 1),
                         "droplet_area_um2": d, "nucleus_area_um2": a,
                         "solidity": 0.92, "ne_contrast": 1.4,
                         "frac_of_droplet": a / d, "truth": truth})


def _recall(res: pd.DataFrame) -> Tuple[int, int, int]:
    tp = int(((res.flag != "") & (res.truth != "")).sum())
    fn = int(((res.flag == "") & (res.truth != "")).sum())
    fp = int(((res.flag != "") & (res.truth == "")).sum())
    return tp, fn, fp


print("1. droplet-size conditioning — does it matter?")
pop_wide = make_synthetic_population(droplet_spread=True)
pop_narrow = make_synthetic_population(droplet_spread=False)
for label, pop in (("wide droplet range", pop_wide), ("narrow range", pop_narrow)):
    for cond in (False, True):
        tp, fn, fp = _recall(scan_size_outliers(pop, cfg, condition_on_droplet=cond))
        print(f"   {label:>18} | conditioned={str(cond):>5}: "
              f"found {tp}/{tp + fn}, {fp} false alarms")

print("\n2. contamination limit — recall collapses as failures become the norm")
print(f"   {'frag %':>7}{'found':>9}{'FP':>5}{'% flagged':>11}")
for frac in (0.05, 0.10, 0.20, 0.33, 0.45):
    res = scan_size_outliers(
        make_synthetic_population(frag_frac=frac, merge_frac=0.0), cfg)
    tp, fn, fp = _recall(res)
    print(f"   {frac*100:>6.0f}%{f'{tp}/{tp+fn}':>9}{fp:>5}"
          f"{res.pct_flagged_this_t.iloc[0]:>11.1f}")

print("\n3. absolute-prior scan stays useful where the population scan does not")
heavy = make_synthetic_population(frag_frac=0.45, merge_frac=0.0)
pop_res = scan_size_outliers(heavy, cfg)
abs_res = scan_absolute_prior(heavy, cfg, min_um2=100.0, max_um2=600.0)
print(f"   population scan flags {int((pop_res.flag != '').sum())} of "
      f"{int((heavy.truth != '').sum())} injected failures")
n_abs_caught = int(((abs_res.abs_flag != "") & (abs_res.truth != "")).sum())
print(f"   absolute-prior scan flags {n_abs_caught} of them "
      f"(external reference, so contamination-proof)")

assert _recall(scan_size_outliers(pop_wide, cfg))[0] > 0, \
    "conditioned scan found nothing on the wide-droplet population"
assert _recall(scan_size_outliers(pop_wide, cfg, condition_on_droplet=False))[0] <= \
       _recall(scan_size_outliers(pop_wide, cfg))[0], \
    "raw-area scan should not beat the droplet-conditioned scan on a wide range"
print("\nscanner self-test passed.")

### 8f. Run the scanners

`RUN_AUDIT` (§8b) produces the `audit` table this consumes, or point it at
`all_droplet_metrics.csv` from §10b once proposals exist. The gallery needs
the proposal `.npz` files, so it only works after §10b has run.

In [ ]:
RUN_OUTLIER_SCAN = False        # ← flip to True once audit or proposals exist

if RUN_OUTLIER_SCAN:
    metrics_path = gs_paths.proposals_dir / "all_droplet_metrics.csv"
    if metrics_path.exists():
        metrics = pd.read_csv(metrics_path)
        source = "proposals"
    else:
        metrics = pd.read_csv(gs_paths.qc_dir / "detector_ab_audit.csv")
        metrics = metrics.rename(columns={"ws_area_um2": "nucleus_area_um2",
                                          "ws_solidity": "solidity",
                                          "ws_ne_contrast": "ne_contrast",
                                          "ws_frac_of_droplet": "frac_of_droplet"})
        metrics["droplet_id"] = metrics.droplet_idx + 1
        source = "proposals"

    flagged = scan_size_outliers(metrics, cfg)
    print("per-timepoint report — read pct_flagged before the flags themselves\n")
    display(outlier_report(flagged))

    absolute = scan_absolute_prior(flagged, cfg)
    n_abs = int((absolute.abs_flag != "").sum())
    print(f"\nabsolute-prior scan (100-600 µm²): {n_abs} outside range")
    if n_abs:
        display(absolute[absolute.abs_flag != ""]
                [["t", "z", "droplet_id", "nucleus_area_um2", "abs_flag",
                  "abs_ratio", "flag", "diagnosis"]].head(20).round(2))

    cols = ["t", "z", "droplet_id", "nucleus_area_um2", "droplet_area_um2",
            "z_size", "flag", "diagnosis"]
    print("\nmost extreme flagged nuclei:")
    display(flagged[flagged.flag != ""][cols].head(20).round(2))

    flagged.to_csv(gs_paths.qc_dir / "outlier_scan.csv", index=False)
    plot_outlier_gallery(flagged, gs_paths, cfg, n=12, source=source)
else:
    print("RUN_OUTLIER_SCAN is False — skipping.")

### 8g. z-consistency scan (contamination-proof)

Slower — it segments three planes per droplet — so it is a separate switch.
Run it whenever `outlier_report` says `AMBIGUOUS`, and on at least one late
timepoint regardless, because that is where the population scan is most likely
to be quiet for the wrong reason.

In [ ]:
RUN_Z_CONSISTENCY = False       # ← flip to True on Cheaha

if RUN_Z_CONSISTENCY:
    hyperstack = load_hyperstack(gs_paths.image_path)
    zc = scan_z_consistency(hyperstack, t=8, z_centre=16, cfg=cfg,
                            dz=1, max_droplets=25)
    print(f"{int((zc.z_flag == 'dip').sum())} droplets lose >35% of their area "
          f"on the centre plane vs. their own neighbours")
    display(zc.head(12).round(2))
    zc.to_csv(gs_paths.qc_dir / "z_consistency_scan.csv", index=False)
    zc["flag"] = zc.z_flag
    plot_outlier_gallery(zc, gs_paths, cfg, n=9, flag_col="flag")
else:
    print("RUN_Z_CONSISTENCY is False — skipping.")

## 9. Sampling plan for the gold standard

A gold standard is defined by its **sampling scheme**, not by its size. Ad-hoc
annotation of whatever looked interesting produces a set that cannot support a
claim about the model's performance on the dataset.

The scheme here:

* **Stratified by timepoint** — nuclear appearance changes qualitatively across
  the timecourse (t = 0–2 NPC puncta only; t = 3–6 partial ring; t = 7–9
  complete ring, bright interior). Uniform sampling would over-weight whichever
  phase has more planes.
* **Focus-restricted in z** — planes below `sample_z_floor` carry coverslip
  artefacts that produce bright round objects with no nucleus. Sampling them
  would build a gold standard whose hardest cases are artefacts.
* **Seeded** — `sample_seed` is in the config and therefore in the run hash.
  The same config regenerates the same sample; a different config gets a
  different run directory rather than silently mixing.
* **Recorded before annotation** — the plane list is written to disk first.
  Choosing planes after seeing which ones the model handles well is how a
  validation set stops being one.

In [ ]:
def build_sampling_plan(T: int, Z: int, cfg: GoldStandardConfig) -> pd.DataFrame:
    """Stratified, seeded (t, z) plane list. Deterministic for a given config."""
    prng = np.random.default_rng(cfg.sample_seed)
    z_hi = cfg.sample_z_ceiling if cfg.sample_z_ceiling is not None else Z - 1
    z_pool = np.arange(cfg.sample_z_floor, z_hi + 1)
    if z_pool.size == 0:
        raise ValueError(f"empty z pool: floor={cfg.sample_z_floor}, ceiling={z_hi}, Z={Z}")

    rows = []
    for t in range(T):
        phase = "early" if t <= 2 else ("mid" if t <= 6 else "late")
        k = min(cfg.sample_planes_per_timepoint, z_pool.size)
        for z in sorted(prng.choice(z_pool, size=k, replace=False)):
            rows.append({"sample_id": f"t{t:02d}_z{int(z):02d}",
                         "t": t, "z": int(z), "phase": phase,
                         "status": "planned"})
    plan = pd.DataFrame(rows)
    plan.attrs["seed"] = cfg.sample_seed
    return plan


def freeze_sampling_plan(plan: pd.DataFrame, gs_paths: GoldStandardPaths) -> Path:
    """Write the plan once. Refuses to overwrite a plan that already exists."""
    out = gs_paths.run_dir / "sampling_plan.csv"
    if out.exists():
        existing = pd.read_csv(out)
        same = existing[["t", "z"]].equals(plan[["t", "z"]].reset_index(drop=True))
        print(f"plan already frozen at {out} — {'identical' if same else 'DIFFERENT'}")
        if not same:
            raise RuntimeError(
                "A frozen sampling plan exists and differs from the one just built. "
                "Change the config (new run id) rather than editing the plan in place.")
        return out
    plan.to_csv(out, index=False)
    print(f"frozen {len(plan)} planes → {out}")
    return out

## 10. Label proposal assembly

Composes the 4-channel multi-label stack for one plane, in the Vulcan
convention (`0 Background · 1 Droplet · 2 NPC · 3 Nucleus`, classes overlap —
a nucleus is also droplet).

**Where `UNANNOTATED` goes.** This is the part that makes the output a
validation set rather than another round of automatic labels:

| Situation | Nucleus channel |
|---|---|
| watershed accepted **and** Stage-2 passed | `1` inside, `0` elsewhere in the droplet |
| watershed rejected, or Stage-2 failed | `255` over the whole droplet interior |
| Stage-2 abstained (t < `gate_min_timepoint`) | `255` over the whole droplet interior |
| outside every droplet | `0` |

An uncertain droplet is marked *unknown*, never *empty*. Scoring a model
against a guess is worse than not scoring it there at all, and §13 honours
these regions as ignore-regions.

In [ ]:
PROPOSAL_METRIC_COLUMNS = [
    "droplet_id", "t", "z", "verdict", "droplet_area_um2", "droplet_circ",
    "centroid_y", "centroid_x", "nucleus_area_um2", "frac_of_droplet",
    "solidity", "ne_contrast", "geom_reject", "stage2", "n_puncta",
    "ring_contrast", "coloc_frac",
]


def build_label_proposal(hyperstack, t: int, z: int, cfg: GoldStandardConfig
                         ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, pd.DataFrame]:
    """
    Returns (labels (4,H,W) uint8 in {0,1,255},
             droplet_instances (H,W) uint16,
             nucleus_instances (H,W) uint16,
             candidate_instances (H,W) uint16 — every watershed mask,
                 including gate-rejected ones, so review can accept them,
             per-droplet metrics DataFrame).
    """
    npc_for_droplets = extract_plane(hyperstack, t, z, cfg.ch_npc)
    droplets = detect_droplets(npc_for_droplets, cfg)[:cfg.sample_max_droplets_per_plane]

    nls = extract_plane(hyperstack, t, z, cfg.ch_nls)
    npc_raw = extract_plane(hyperstack, t, z, cfg.ch_npc)       # RAW, re-pulled
    mem_raw = extract_plane(hyperstack, t, z, cfg.ch_membrane)
    H, W = nls.shape

    droplet_ch = np.zeros((H, W), np.uint8)
    nucleus_ch = np.zeros((H, W), np.uint8)
    npc_ch = np.zeros((H, W), np.uint8)
    droplet_inst = np.zeros((H, W), np.uint16)
    nucleus_inst = np.zeros((H, W), np.uint16)
    # every watershed candidate, gate verdict aside — without this a reviewer
    # can reject a mask but never accept one the gates threw away
    candidate_inst = np.zeros((H, W), np.uint16)

    rows = []
    for i, d in enumerate(droplets, start=1):
        bb = d["bbox"]
        r0, c0, r1, c1 = bb
        dm = crop_bbox(d["mask"], bb)
        nls_c, npc_c, mem_c = (crop_bbox(a, bb) for a in (nls, npc_raw, mem_raw))

        droplet_ch[r0:r1, c0:c1][dm] = 1
        droplet_inst[r0:r1, c0:c1][dm] = i

        m_ws, i_ws = segment_nucleus_watershed(nls_c, dm, cfg)
        s2_pass, s2 = stage2_gate(m_ws, npc_c, mem_c, dm, cfg, t_idx=t)
        if m_ws.any():
            candidate_inst[r0:r1, c0:c1][m_ws] = i

        verdict = ("accept" if (i_ws["accepted"] and s2_pass is True)
                   else ("abstain" if s2_pass is None else "reject"))

        if verdict == "accept":
            nucleus_ch[r0:r1, c0:c1][m_ws] = 1
            nucleus_inst[r0:r1, c0:c1][m_ws] = i
            puncta = detect_npc_puncta(npc_c, m_ws, dm, cfg)
            npc_ch[r0:r1, c0:c1][puncta] = 1
        else:
            # unknown, not empty — reviewer resolves, metrics ignore until then
            sub = nucleus_ch[r0:r1, c0:c1]
            sub[dm & (sub == 0)] = UNANNOTATED
            sub2 = npc_ch[r0:r1, c0:c1]
            sub2[dm & (sub2 == 0)] = UNANNOTATED

        rows.append({
            "droplet_id": i, "t": t, "z": z, "verdict": verdict,
            "droplet_area_um2": d["area_um2"], "droplet_circ": d["circ"],
            "centroid_y": d["centroid"][0], "centroid_x": d["centroid"][1],
            "nucleus_area_um2": i_ws["area_um2"],
            "frac_of_droplet": i_ws["frac_of_droplet"],
            "solidity": i_ws["solidity"], "ne_contrast": i_ws["ne_contrast"],
            "geom_reject": i_ws["reject"], "stage2": s2["stage2"],
            "n_puncta": s2["n_puncta"], "ring_contrast": s2["ring_contrast"],
            "coloc_frac": s2["coloc_frac"],
        })

    background = np.where(droplet_ch == 1, 0, 1).astype(np.uint8)
    labels = np.stack([background, droplet_ch, npc_ch, nucleus_ch]).astype(np.uint8)
    # columns declared explicitly so a plane with zero droplets still returns a
    # well-formed (empty) frame instead of a column-less one that breaks
    # every downstream .verdict / .nucleus_area_um2 access
    return (labels, droplet_inst, nucleus_inst, candidate_inst,
            pd.DataFrame(rows, columns=PROPOSAL_METRIC_COLUMNS))


def check_label_invariants(labels: np.ndarray) -> List[str]:
    """Structural rules a valid 4-channel multi-label stack must satisfy."""
    bg, drop, npc, nuc = labels
    problems = []
    known_nuc, known_drop = (nuc != UNANNOTATED), (drop != UNANNOTATED)
    both = known_nuc & known_drop
    if np.any((nuc == 1) & both & (drop != 1)):
        problems.append("nucleus pixels outside the droplet channel")
    if np.any((npc == 1) & known_drop & (drop != 1)):
        problems.append("NPC pixels outside the droplet channel")
    if np.any((bg == 1) & known_drop & (drop == 1)):
        problems.append("background and droplet both set")
    if set(np.unique(labels)) - {0, 1, UNANNOTATED}:
        problems.append(f"unexpected label values: {sorted(set(np.unique(labels)))}")
    return problems


def save_proposal(sample_id: str, t: int, z: int, hyperstack,
                  labels, droplet_inst, nucleus_inst, candidate_inst, metrics,
                  cfg: GoldStandardConfig, gs_paths: GoldStandardPaths) -> Path:
    img = np.stack([np.asarray(hyperstack[t, z, c])
                    for c in (cfg.ch_membrane, cfg.ch_nls, cfg.ch_npc)])
    meta = {"sample_id": sample_id, "t": t, "z": z,
            "run_id": cfg.run_id(), "notebook_version": cfg.notebook_version,
            "dataset": cfg.dataset_name, "pixel_size_um": cfg.pixel_size_um,
            "z_step_um": cfg.z_step_um, "rig": cfg.rig_name,
            "channel_order": ["membrane", "nls", "npc"],
            "class_order": CLASS_NAMES, "unannotated_value": UNANNOTATED,
            "created": datetime.now().isoformat(timespec="seconds"),
            "invariant_problems": check_label_invariants(labels)}
    out = gs_paths.proposals_dir / f"{sample_id}.npz"
    np.savez_compressed(out, image=img.astype(np.uint16), labels=labels,
                        droplet_instances=droplet_inst,
                        nucleus_instances=nucleus_inst,
                        candidate_instances=candidate_inst,
                        meta=json.dumps(meta))
    metrics.to_csv(gs_paths.proposals_dir / f"{sample_id}_droplets.csv", index=False)
    return out

### 10b. Generate proposals for the frozen plan

In [ ]:
RUN_PROPOSALS = False       # ← flip to True on Cheaha

if RUN_PROPOSALS:
    hyperstack = load_hyperstack(gs_paths.image_path)
    T, Z = hyperstack.shape[0], hyperstack.shape[1]
    plan = build_sampling_plan(T, Z, cfg)
    freeze_sampling_plan(plan, gs_paths)

    all_metrics, empty_planes = [], []
    for row in plan.itertuples():
        labels, dinst, ninst, cinst, met = build_label_proposal(
            hyperstack, row.t, row.z, cfg)
        problems = check_label_invariants(labels)
        save_proposal(row.sample_id, row.t, row.z, hyperstack,
                      labels, dinst, ninst, cinst, met, cfg, gs_paths)
        all_metrics.append(met)
        n_unk = int((labels[CLASS_NUCLEUS] == UNANNOTATED).sum())
        n_accept = int((met.verdict == "accept").sum()) if len(met) else 0
        if len(met) == 0:
            empty_planes.append((row.sample_id, row.t, row.z))
        print(f"{row.sample_id}: {len(met):3d} droplets  "
              f"accept={n_accept:3d}  unknown_px={n_unk:>8d}  "
              f"{'OK' if not problems else 'PROBLEMS: ' + '; '.join(problems)}"
              f"{'   <-- NO DROPLETS DETECTED' if len(met) == 0 else ''}")

    pd.concat(all_metrics, ignore_index=True).to_csv(
        gs_paths.proposals_dir / "all_droplet_metrics.csv", index=False)

    if empty_planes:
        sid, t0, z0 = empty_planes[0]
        print(f"\n{len(empty_planes)} of {len(plan)} planes found no droplets. "
              f"Attrition for the first one ({sid}):\n")
        display(diagnose_droplet_detection(
            extract_plane(hyperstack, t0, z0, cfg.ch_npc), cfg))
        print("\nRead the first row that goes to zero — that is the gate to fix.")
        print("If it is 'foreground pixels after threshold', the NPC channel is too")
        print("dim on this plane; droplet detection keys on NPC, and at t=0-1 there")
        print("are only puncta. Try a mid-timecourse plane first, or switch the")
        print("droplet source to the membrane channel.")
else:
    print("RUN_PROPOSALS is False — skipping.")

## 11. napari review

**Run this in `napari_env`, not `ml_env_tf_2.15`.** The proposal `.npz` files
are the handoff — no TensorFlow, no CUDA, nothing from the training stack is
needed to review them.

What the reviewer does per plane:

1. Fix nucleus extent where the watershed stopped early or flooded.
2. Resolve every `UNANNOTATED` droplet — paint the nucleus, or paint the
   droplet interior to 0 to assert "no nucleus here". Whatever is left at 255
   stays an ignore-region; that is a legitimate outcome for a genuinely
   ambiguous droplet, not a failure to finish.
3. Save to `review/<sample_id>.npz` with `reviewer` and `plane_done` set.

The three raw channels are shown as separate image layers because the calls
that matter — is that a cap or a nucleus, is that ring real — are made on the
NPC and membrane channels, not on NLS.

In [ ]:
def launch_napari_review(sample_id: str, gs_paths: GoldStandardPaths):
    """Open one proposal for editing. Returns (viewer, state) — keep both alive."""
    import napari                                     # napari_env only

    d = np.load(gs_paths.proposals_dir / f"{sample_id}.npz", allow_pickle=False)
    img, labels = d["image"], d["labels"].copy()
    meta = json.loads(str(d["meta"]))

    viewer = napari.Viewer(title=f"gold standard — {sample_id}")
    for name, arr, cmap in (("membrane", img[0], "gray"),
                            ("NLS", img[1], "green"),
                            ("NPC", img[2], "magenta")):
        viewer.add_image(arr, name=name, colormap=cmap, blending="additive")
    for idx, name in ((CLASS_DROPLET, "droplet"), (CLASS_NPC, "NPC label"),
                      (CLASS_NUCLEUS, "nucleus")):
        viewer.add_labels(labels[idx].astype(np.uint8), name=name, opacity=0.45)
    print(f"{sample_id}: unknown nucleus px = "
          f"{int((labels[CLASS_NUCLEUS] == UNANNOTATED).sum())}")
    return viewer, {"sample_id": sample_id, "meta": meta,
                    "image": img, "labels": labels}


def save_review(viewer, state, reviewer: str, plane_done: bool,
                gs_paths: GoldStandardPaths, notes: str = "") -> Path:
    """Pull the edited label layers back out of napari and write review/<id>.npz."""
    labels = state["labels"].copy()
    for idx, name in ((CLASS_DROPLET, "droplet"), (CLASS_NPC, "NPC label"),
                      (CLASS_NUCLEUS, "nucleus")):
        labels[idx] = np.asarray(viewer.layers[name].data, dtype=np.uint8)
    labels[CLASS_BACKGROUND] = np.where(labels[CLASS_DROPLET] == 1, 0, 1).astype(np.uint8)

    problems = check_label_invariants(labels)
    meta = dict(state["meta"])
    meta.update({"reviewer": reviewer, "plane_done": bool(plane_done),
                 "reviewed_at": datetime.now().isoformat(timespec="seconds"),
                 "notes": notes, "invariant_problems": problems,
                 "unknown_px": {CLASS_NAMES[i]: int((labels[i] == UNANNOTATED).sum())
                                for i in range(4)}})
    out = gs_paths.review_dir / f"{state['sample_id']}.npz"
    np.savez_compressed(out, image=state["image"], labels=labels,
                        meta=json.dumps(meta))
    print(f"saved {out}" + (f"\n  PROBLEMS: {problems}" if problems else ""))
    return out

## 11b. Review on Cheaha, without napari

napari needs a GUI and does not run over the VS Code tunnel. This section is
the substitute — but it is deliberately *not* a napari clone, because most of
what §11 asks for is not painting:

| Review action | Needs pixel editing? | Handled here |
|---|---|---|
| resolve an `UNANNOTATED` droplet — is there a nucleus or not | no, it's a verdict | **yes** |
| accept a mask the gates rejected | no | **yes** |
| reject a mask the gates accepted | no | **yes** |
| mask slightly too tight or too loose | a uniform nudge covers most cases | **yes** (`adjust`, ±px) |
| boundary wrong in one direction only, or two nuclei to separate | yes | **no** — flag `needs_edit`, do it in napari locally |

So the division of labour is: triage everything on Cheaha, and export the
small residue that genuinely needs a brush to `needs_edit.csv`, pull those
planes to StarForge over Globus, and paint only those in `napari_env`.

**Prerequisite.** The proposal `.npz` now also stores `candidate_instances` —
the watershed mask for *every* droplet, including ones the gates rejected.
Without it you could reject a candidate but never accept one, because a
rejected droplet's mask was previously discarded and only its droplet
interior survived as 255. Proposals written before this change fall back to
`nucleus_instances`, so accepting a gate-rejected droplet will not work on
them — regenerate proposals if you need that.

In [ ]:
VERDICTS = ("accept", "no_nucleus", "ambiguous", "needs_edit")
VERDICT_HELP = {
    "accept":     "mask is right (after any adjust) -> nucleus = 1",
    "no_nucleus": "no nucleus in this droplet     -> nucleus = 0 across interior",
    "ambiguous":  "cannot tell                    -> stays 255, ignored by metrics",
    "needs_edit": "real nucleus, mask needs a brush -> stays 255, exported for napari",
}


def load_proposal(sample_id: str, gs_paths: GoldStandardPaths,
                  source: str = "proposals") -> dict:
    p = getattr(gs_paths, f"{source}_dir") / f"{sample_id}.npz"
    if not p.exists():
        raise FileNotFoundError(p)
    d = np.load(p, allow_pickle=False)
    out = {"image": d["image"], "labels": d["labels"].copy(),
           "droplet_instances": d["droplet_instances"],
           "nucleus_instances": d["nucleus_instances"],
           "meta": json.loads(str(d["meta"])), "sample_id": sample_id}
    out["candidate_instances"] = (d["candidate_instances"] if
                                  "candidate_instances" in d.files
                                  else d["nucleus_instances"])
    if "candidate_instances" not in d.files:
        print(f"{sample_id}: no candidate_instances (old proposal) — "
              f"gate-rejected droplets cannot be accepted from this file.")
    return out


def _droplet_window(prop: dict, did: int, pad_px: int):
    ys, xs = np.where(prop["droplet_instances"] == did)
    if ys.size == 0:
        return None
    H, W = prop["droplet_instances"].shape
    return (max(int(ys.min()) - pad_px, 0), max(int(xs.min()) - pad_px, 0),
            min(int(ys.max()) + pad_px, H), min(int(xs.max()) + pad_px, W))


def _stretch(a, lo_pct=1, hi_pct=99.5):
    a = np.asarray(a, np.float32)
    lo, hi = np.percentile(a, [lo_pct, hi_pct])
    return np.clip((a - lo) / (hi - lo + 1e-8), 0, 1)


def adjust_mask(mask: np.ndarray, adjust_px: int,
                confine_to: Optional[np.ndarray] = None) -> np.ndarray:
    """Uniform dilate (+) or erode (-) in px. Confined to the droplet interior."""
    if adjust_px == 0 or not mask.any():
        out = mask
    elif adjust_px > 0:
        out = morphology.binary_dilation(mask, morphology.disk(int(adjust_px)))
    else:
        out = morphology.binary_erosion(mask, morphology.disk(int(-adjust_px)))
    return (out & confine_to) if confine_to is not None else out


def render_droplet(prop: dict, did: int, cfg: GoldStandardConfig,
                   adjust_px: int = 0, pad_um: float = 3.0, figsize=(13, 3.6)):
    """Three channels plus the candidate mask, for one droplet."""
    win = _droplet_window(prop, did, cfg.px(pad_um))
    if win is None:
        print(f"droplet {did} not found"); return None
    r0, c0, r1, c1 = win
    img = prop["image"][:, r0:r1, c0:c1]
    dm = prop["droplet_instances"][r0:r1, c0:c1] == did
    cand = prop["candidate_instances"][r0:r1, c0:c1] == did
    cand_adj = adjust_mask(cand, adjust_px, confine_to=dm)
    nuc_ch = prop["labels"][CLASS_NUCLEUS][r0:r1, c0:c1]

    state = ("proposed nucleus" if np.any((nuc_ch == 1) & dm)
             else ("UNANNOTATED" if np.any((nuc_ch == UNANNOTATED) & dm) else "background"))

    fig, ax = plt.subplots(1, 4, figsize=figsize)
    for a, (ch, name) in zip(ax[:3], ((1, "NLS"), (2, "NPC"), (0, "membrane"))):
        a.imshow(_stretch(img[ch]), cmap="gray")
        a.contour(dm, levels=[0.5], colors="#ffd700", linewidths=0.7)
        if cand_adj.any():
            a.contour(cand_adj, levels=[0.5], colors="c", linewidths=1.2)
        a.set_title(name, fontsize=9); a.axis("off")

    overlay = np.zeros((*dm.shape, 3), np.float32)
    overlay[..., 1] = _stretch(img[1])
    overlay[..., 0] = _stretch(img[2]) * 0.9
    ax[3].imshow(overlay)
    if cand_adj.any():
        ax[3].contour(cand_adj, levels=[0.5], colors="w", linewidths=1.4)
    frac = cand_adj.sum() / max(dm.sum(), 1)
    ax[3].set_title(f"NLS+NPC · {cfg.px_to_um2(int(cand_adj.sum())):.0f} µm² "
                    f"({frac:.2f} of droplet)", fontsize=9)
    ax[3].axis("off")
    fig.suptitle(f"{prop['sample_id']}  droplet {did}  ·  proposal: {state}"
                 f"{f'  ·  adjust {adjust_px:+d} px' if adjust_px else ''}",
                 fontsize=10)
    plt.tight_layout(); plt.show()
    return {"area_um2": float(cfg.px_to_um2(int(cand_adj.sum()))), "frac": float(frac)}


def contact_sheet(sample_id: str, gs_paths: GoldStandardPaths,
                  cfg: GoldStandardConfig, only_unresolved: bool = True,
                  n: int = 24, ncol: int = 6, pad_um: float = 2.0,
                  source: str = "proposals"):
    """Whole-plane overview — decide where to spend attention before triaging."""
    prop = load_proposal(sample_id, gs_paths, source)
    dids = [d for d in np.unique(prop["droplet_instances"]) if d > 0]
    if only_unresolved:
        nuc = prop["labels"][CLASS_NUCLEUS]
        dids = [d for d in dids
                if np.any((nuc == UNANNOTATED) & (prop["droplet_instances"] == d))]
    dids = dids[:n]
    if not dids:
        print(f"{sample_id}: nothing to show"
              f"{' (no unresolved droplets)' if only_unresolved else ''}")
        return []
    nrow = int(np.ceil(len(dids) / ncol))
    fig, ax = plt.subplots(nrow, ncol, figsize=(2.3 * ncol, 2.5 * nrow))
    ax = np.atleast_1d(ax).ravel()
    for k, did in enumerate(dids):
        win = _droplet_window(prop, int(did), cfg.px(pad_um))
        r0, c0, r1, c1 = win
        ax[k].imshow(_stretch(prop["image"][1, r0:r1, c0:c1]), cmap="gray")
        dm = prop["droplet_instances"][r0:r1, c0:c1] == did
        cand = prop["candidate_instances"][r0:r1, c0:c1] == did
        ax[k].contour(dm, levels=[0.5], colors="#ffd700", linewidths=0.6)
        if cand.any():
            ax[k].contour(cand, levels=[0.5], colors="c", linewidths=1.0)
        ax[k].set_title(f"d{int(did)}", fontsize=8); ax[k].axis("off")
    for a in ax[len(dids):]:
        a.axis("off")
    fig.suptitle(f"{sample_id} — {'unresolved' if only_unresolved else 'all'} "
                 f"droplets ({len(dids)} shown)", fontsize=11)
    plt.tight_layout(); plt.show()
    return [int(d) for d in dids]

### 11c. Triage

`TriageSession` steps through droplets one at a time with buttons. It
autosaves to `review/<sample_id>_verdicts.csv` after every decision, so a
dropped tunnel or a walltime kill costs you nothing — reopen and it resumes
where you left off.

If `ipywidgets` is missing, fall back to `verdict_template()`, which writes
the same CSV with every droplet set to `ambiguous` for you to edit by hand,
and `render_droplet()` to look at them one at a time. `apply_verdicts` reads
the CSV either way, so the two paths are interchangeable.

Install widgets if needed (they work in the VS Code notebook UI):
`conda install -n ml_env_tf_2.15 ipywidgets` — no GUI, no X forwarding.

In [ ]:
def verdict_csv_path(sample_id: str, gs_paths: GoldStandardPaths) -> Path:
    return gs_paths.review_dir / f"{sample_id}_verdicts.csv"


def verdict_template(sample_id: str, gs_paths: GoldStandardPaths,
                     cfg: GoldStandardConfig, overwrite: bool = False) -> Path:
    """Blank verdict sheet, one row per droplet, everything 'ambiguous'."""
    out = verdict_csv_path(sample_id, gs_paths)
    if out.exists() and not overwrite:
        print(f"{out} exists — pass overwrite=True to reset"); return out
    prop = load_proposal(sample_id, gs_paths)
    dids = [int(d) for d in np.unique(prop["droplet_instances"]) if d > 0]
    nuc = prop["labels"][CLASS_NUCLEUS]
    rows = []
    for d in dids:
        dm = prop["droplet_instances"] == d
        proposed = "nucleus" if np.any((nuc == 1) & dm) else (
            "unannotated" if np.any((nuc == UNANNOTATED) & dm) else "background")
        cand_px = int((prop["candidate_instances"] == d).sum())
        rows.append({"sample_id": sample_id, "droplet_id": d,
                     "proposal": proposed,
                     "candidate_area_um2": round(float(cfg.px_to_um2(cand_px)), 1),
                     "verdict": "accept" if proposed == "nucleus" else "ambiguous",
                     "adjust_px": 0, "note": ""})
    df = pd.DataFrame(rows, columns=["sample_id", "droplet_id", "proposal",
                                     "candidate_area_um2", "verdict",
                                     "adjust_px", "note"])
    df.to_csv(out, index=False)
    if df.empty:
        print(f"{sample_id}: NO DROPLETS in this proposal — nothing to review. "
              f"Run diagnose_droplet_detection on this plane (see 4).")
        return out
    print(f"wrote {len(df)} rows -> {out}")
    print("verdict must be one of:", ", ".join(VERDICTS))
    for k, v in VERDICT_HELP.items():
        print(f"   {k:<11} {v}")
    return out


class TriageSession:
    """Button-driven per-droplet triage. Autosaves after every decision."""

    def __init__(self, sample_id: str, gs_paths: GoldStandardPaths,
                 cfg: GoldStandardConfig, unresolved_first: bool = True,
                 pad_um: float = 3.0):
        self.sid, self.paths, self.cfg, self.pad_um = sample_id, gs_paths, cfg, pad_um
        self.prop = load_proposal(sample_id, gs_paths)
        path = verdict_csv_path(sample_id, gs_paths)
        if not path.exists():
            verdict_template(sample_id, gs_paths, cfg)
        self.df = pd.read_csv(path)
        if self.df.empty:
            raise ValueError(
                f"{sample_id}: verdict sheet is empty — the proposal found no "
                f"droplets. Run diagnose_droplet_detection on this plane.")
        self.df = self.df.set_index("droplet_id")
        order = list(self.df.index)
        if unresolved_first:
            order.sort(key=lambda d: (self.df.at[d, "proposal"] != "unannotated", d))
        self.order = order
        self.i = 0

    # -- persistence -----------------------------------------------------
    def save(self):
        self.df.reset_index().to_csv(verdict_csv_path(self.sid, self.paths),
                                     index=False)

    def set(self, verdict: str, adjust_px: Optional[int] = None, note: str = ""):
        if verdict not in VERDICTS:
            raise ValueError(f"verdict must be one of {VERDICTS}")
        d = self.order[self.i]
        self.df.at[d, "verdict"] = verdict
        if adjust_px is not None:
            self.df.at[d, "adjust_px"] = int(adjust_px)
        if note:
            self.df.at[d, "note"] = note
        self.save()

    def progress(self) -> str:
        done = int((self.df.verdict != "ambiguous").sum())
        return f"{done}/{len(self.df)} decided · {self.i + 1} of {len(self.order)}"

    # -- non-widget use --------------------------------------------------
    def show(self, adjust_px: Optional[int] = None):
        d = self.order[self.i]
        a = int(self.df.at[d, "adjust_px"]) if adjust_px is None else adjust_px
        render_droplet(self.prop, d, self.cfg, adjust_px=a, pad_um=self.pad_um)
        print(f"droplet {d} · current verdict = {self.df.at[d, 'verdict']} "
              f"· adjust {a:+d} px · {self.progress()}")

    def next(self, step: int = 1):
        self.i = int(np.clip(self.i + step, 0, len(self.order) - 1))
        self.show()

    # -- widget use ------------------------------------------------------
    def launch(self):
        try:
            import ipywidgets as W
            from IPython.display import display as _display, clear_output
        except ImportError:
            print("ipywidgets not installed. Use the manual path instead:\n"
                  "    s = TriageSession(sample_id, gs_paths, cfg)\n"
                  "    s.show()                       # look\n"
                  "    s.set('accept', adjust_px=1)   # decide\n"
                  "    s.next()                       # advance\n"
                  "or edit the verdicts CSV directly and run apply_verdicts().")
            return None

        out = W.Output()
        adj = W.IntSlider(value=0, min=-5, max=5, description="adjust px",
                          continuous_update=False)
        note = W.Text(description="note", placeholder="optional")
        status = W.HTML()

        def draw():
            d = self.order[self.i]
            adj.value = int(self.df.at[d, "adjust_px"])
            with out:
                clear_output(wait=True)
                render_droplet(self.prop, d, self.cfg, adjust_px=adj.value,
                               pad_um=self.pad_um)
            status.value = (f"<b>droplet {d}</b> — proposal "
                            f"<code>{self.df.at[d, 'proposal']}</code>, verdict "
                            f"<code>{self.df.at[d, 'verdict']}</code><br>"
                            f"{self.progress()}")

        def decide(v):
            def _cb(_):
                self.set(v, adjust_px=adj.value, note=note.value)
                note.value = ""
                if self.i < len(self.order) - 1:
                    self.i += 1
                draw()
            return _cb

        def step(n):
            def _cb(_):
                self.i = int(np.clip(self.i + n, 0, len(self.order) - 1)); draw()
            return _cb

        buttons = [W.Button(description=v, button_style=s, tooltip=VERDICT_HELP[v])
                   for v, s in zip(VERDICTS, ("success", "danger", "warning", "info"))]
        for b, v in zip(buttons, VERDICTS):
            b.on_click(decide(v))
        prev_b = W.Button(description="< prev"); prev_b.on_click(step(-1))
        next_b = W.Button(description="next >"); next_b.on_click(step(+1))
        redraw = W.Button(description="redraw"); redraw.on_click(lambda _: draw())
        adj.observe(lambda ch: draw(), names="value")

        _display(W.VBox([status, W.HBox(buttons),
                         W.HBox([prev_b, next_b, adj, redraw]), note, out]))
        draw()
        return self

### 11d. Apply verdicts and hand off

`apply_verdicts` turns the CSV into a `review/<sample_id>.npz` that
`commit_reviewed_plane` (§12) accepts unchanged — the Cheaha path and the
napari path converge here.

NPC puncta are **recomputed** for any droplet accepted after being
gate-rejected, because the proposal never wrote puncta for those (the puncta
detector is anchored to the nucleus boundary, which did not exist yet).

`plane_done` is set only when nothing is left as `ambiguous` — but
`needs_edit` droplets do *not* block it. They stay 255 and are exported to
`needs_edit.csv`; a plane can be committed with them present, and §13 treats
them as ignore-regions rather than as errors.

In [ ]:
def apply_verdicts(sample_id: str, gs_paths: GoldStandardPaths,
                   cfg: GoldStandardConfig, reviewer: str,
                   notes: str = "") -> Optional[Path]:
    prop = load_proposal(sample_id, gs_paths)
    path = verdict_csv_path(sample_id, gs_paths)
    if not path.exists():
        print(f"no verdicts at {path}"); return None
    v = pd.read_csv(path)
    if v.empty:
        print(f"{sample_id}: verdict sheet is empty (no droplets) — skipping")
        return None
    bad = set(v.verdict.unique()) - set(VERDICTS)
    if bad:
        print(f"unknown verdicts {sorted(bad)} — must be one of {VERDICTS}")
        return None

    labels = prop["labels"].copy()
    dinst, cand = prop["droplet_instances"], prop["candidate_instances"]
    img = prop["image"]
    counts = {k: 0 for k in VERDICTS}

    for row in v.itertuples():
        did = int(row.droplet_id)
        dm = dinst == did
        if not dm.any():
            continue
        counts[row.verdict] += 1
        nuc_ch, npc_ch = labels[CLASS_NUCLEUS], labels[CLASS_NPC]

        if row.verdict == "accept":
            mask = adjust_mask(cand == did, int(row.adjust_px), confine_to=dm)
            if not mask.any():
                print(f"  droplet {did}: accepted but empty mask — left unannotated")
                continue
            nuc_ch[dm] = 0
            nuc_ch[mask] = 1
            ys, xs = np.where(dm)
            r0, c0, r1, c1 = ys.min(), xs.min(), ys.max() + 1, xs.max() + 1
            puncta = detect_npc_puncta(img[2, r0:r1, c0:c1].astype(np.float32),
                                       mask[r0:r1, c0:c1], dm[r0:r1, c0:c1], cfg)
            sub = npc_ch[r0:r1, c0:c1]
            sub[dm[r0:r1, c0:c1]] = 0
            sub[puncta] = 1
        elif row.verdict == "no_nucleus":
            nuc_ch[dm] = 0
            npc_ch[dm] = 0
        else:                                   # ambiguous / needs_edit
            nuc_ch[dm] = UNANNOTATED
            npc_ch[dm] = UNANNOTATED

    labels[CLASS_BACKGROUND] = np.where(labels[CLASS_DROPLET] == 1, 0, 1).astype(np.uint8)
    problems = check_label_invariants(labels)
    plane_done = counts["ambiguous"] == 0

    meta = dict(prop["meta"])
    meta.update({"reviewer": reviewer, "plane_done": bool(plane_done),
                 "reviewed_at": datetime.now().isoformat(timespec="seconds"),
                 "notes": notes, "review_tool": "notebook_triage_11b",
                 "verdict_counts": counts, "invariant_problems": problems,
                 "unknown_px": {CLASS_NAMES[i]: int((labels[i] == UNANNOTATED).sum())
                                for i in range(4)}})
    out = gs_paths.review_dir / f"{sample_id}.npz"
    np.savez_compressed(out, image=img, labels=labels, meta=json.dumps(meta))

    print(f"{sample_id}: " + "  ".join(f"{k}={counts[k]}" for k in VERDICTS))
    print(f"  plane_done={plane_done}"
          f"{'' if plane_done else '  (ambiguous droplets remain — not committable)'}")
    if problems:
        print(f"  INVARIANT PROBLEMS: {problems}")
    if counts["needs_edit"]:
        ne = v[v.verdict == "needs_edit"].copy()
        ne["sample_id"] = sample_id
        p = gs_paths.review_dir / "needs_edit.csv"
        ne.to_csv(p, mode="a", header=not p.exists(), index=False)
        print(f"  {counts['needs_edit']} droplets exported to {p} for napari")
    return out


def review_status(gs_paths: GoldStandardPaths) -> pd.DataFrame:
    """Where every planned plane stands."""
    rows = []
    plan_p = gs_paths.run_dir / "sampling_plan.csv"
    plan = pd.read_csv(plan_p) if plan_p.exists() else pd.DataFrame()
    for sid in (plan.sample_id.tolist() if len(plan) else
                sorted(p.stem for p in gs_paths.proposals_dir.glob("t*_z*.npz"))):
        vp = verdict_csv_path(sid, gs_paths)
        r = {"sample_id": sid,
             "proposal": (gs_paths.proposals_dir / f"{sid}.npz").exists(),
             "verdicts": vp.exists(),
             "reviewed": (gs_paths.review_dir / f"{sid}.npz").exists(),
             "committed": (gs_paths.committed_dir / f"{sid}.npz").exists()}
        if vp.exists():
            try:
                v = pd.read_csv(vp)
            except pd.errors.EmptyDataError:
                v = pd.DataFrame(columns=["verdict"])
            r["n_droplets"] = len(v)
            for k in VERDICTS:
                r[k] = int((v.verdict == k).sum())
        rows.append(r)
    return pd.DataFrame(rows)

### 11e. Run the triage

In [ ]:
RUN_TRIAGE = False          # ← flip to True on Cheaha

if RUN_TRIAGE:
    SAMPLE_ID = "t04_z16"    # ← the plane to review
    REVIEWER = getpass.getuser()

    contact_sheet(SAMPLE_ID, gs_paths, cfg, only_unresolved=True)
    session = TriageSession(SAMPLE_ID, gs_paths, cfg)
    session.launch()         # falls back to printed instructions without ipywidgets
else:
    print("RUN_TRIAGE is False — skipping.")

In [ ]:
APPLY_VERDICTS = False      # ← flip to True once the plane is triaged

if APPLY_VERDICTS:
    apply_verdicts(SAMPLE_ID, gs_paths, cfg, reviewer=REVIEWER)
    display(review_status(gs_paths))
    commit_reviewed_plane(SAMPLE_ID, cfg, gs_paths)
else:
    print("APPLY_VERDICTS is False — skipping.")

## 12. Commit

A reviewed plane enters the gold standard only if it passes every check.
Committing is deliberately all-or-nothing per plane: a half-reviewed plane in
the validation set produces metrics that look precise and are not.

1. `plane_done` is set by the reviewer.
2. `check_label_invariants` is clean.
3. At least one droplet is resolved (a plane of pure 255 contributes nothing).
4. The proposal config hash matches the live config.

Committed planes are copied to `committed/` and appended to
`committed/manifest.jsonl` with an SHA-1 of the label array, so a later
question of "were these the labels we scored against?" has an answer.

In [ ]:
def commit_reviewed_plane(sample_id: str, cfg: GoldStandardConfig,
                          gs_paths: GoldStandardPaths, force: bool = False) -> bool:
    src = gs_paths.review_dir / f"{sample_id}.npz"
    if not src.exists():
        print(f"{sample_id}: no review file"); return False

    d = np.load(src, allow_pickle=False)
    labels = d["labels"]
    meta = json.loads(str(d["meta"]))

    fails = []
    if not meta.get("plane_done"):
        fails.append("plane_done is False")
    problems = check_label_invariants(labels)
    if problems:
        fails.append("invariants: " + "; ".join(problems))
    nuc = labels[CLASS_NUCLEUS]
    if not np.any(nuc != UNANNOTATED):
        fails.append("no resolved pixels in the nucleus channel")
    if meta.get("run_id") != cfg.run_id():
        fails.append(f"run_id mismatch: proposal={meta.get('run_id')} live={cfg.run_id()}")
    if fails and not force:
        print(f"{sample_id}: REJECTED — " + " | ".join(fails)); return False

    label_sha = hashlib.sha1(np.ascontiguousarray(labels).tobytes()).hexdigest()
    dst = gs_paths.committed_dir / f"{sample_id}.npz"
    np.savez_compressed(dst, image=d["image"], labels=labels,
                        meta=json.dumps({**meta, "label_sha1": label_sha,
                                         "committed_at": datetime.now().isoformat(
                                             timespec="seconds"),
                                         "forced": bool(fails and force)}))
    with open(gs_paths.committed_dir / "manifest.jsonl", "a") as fh:
        fh.write(json.dumps({
            "sample_id": sample_id, "t": meta.get("t"), "z": meta.get("z"),
            "reviewer": meta.get("reviewer"), "label_sha1": label_sha,
            "unknown_px": meta.get("unknown_px"),
            "nucleus_px": int((nuc == 1).sum()),
            "committed_at": datetime.now().isoformat(timespec="seconds"),
            "forced": bool(fails and force), "fails": fails}) + "\n")
    print(f"{sample_id}: committed  (sha1 {label_sha[:10]})")
    return True


def load_committed(gs_paths: GoldStandardPaths) -> List[dict]:
    out = []
    for p in sorted(gs_paths.committed_dir.glob("*.npz")):
        d = np.load(p, allow_pickle=False)
        out.append({"path": p, "labels": d["labels"], "image": d["image"],
                    "meta": json.loads(str(d["meta"]))})
    return out


def gold_standard_summary(gs_paths: GoldStandardPaths, cfg: GoldStandardConfig
                          ) -> pd.DataFrame:
    rows = []
    for c in load_committed(gs_paths):
        nuc = c["labels"][CLASS_NUCLEUS]
        inst = measure.label(nuc == 1, connectivity=2)
        areas = [r.area for r in measure.regionprops(inst)]
        rows.append({
            "sample_id": c["meta"]["sample_id"], "t": c["meta"]["t"], "z": c["meta"]["z"],
            "reviewer": c["meta"].get("reviewer"), "n_nuclei": int(inst.max()),
            "median_area_um2": float(cfg.px_to_um2(np.median(areas))) if areas else np.nan,
            "unknown_frac": float((nuc == UNANNOTATED).mean()),
        })
    return pd.DataFrame(rows).sort_values(["t", "z"]) if rows else pd.DataFrame()

## 13. Scoring Vulcan 1.1 against the gold standard

Semantic Dice alone will not surface the failure this notebook exists to fix.
A model that returns three fragments covering 60 % of a nucleus and a model
that returns one clean mask covering 60 % score identically on Dice. The
instance metrics below separate them.

| Metric | Question |
|---|---|
| Dice, IoU (per class) | overall pixel agreement |
| precision / recall / F1 @ IoU ≥ 0.5 | are nuclei found as objects |
| **split rate** | GT nuclei covered by ≥ 2 predicted instances — *the fragmentation metric* |
| **merge rate** | predicted instances covering ≥ 2 GT nuclei |
| **area bias** | median (pred − GT) / GT over matched pairs — *the incomplete-extent metric* |
| boundary F1 @ tolerance | is the envelope in the right place |

Every metric honours `UNANNOTATED` as an ignore-region: pixels are dropped
from Dice, and predicted objects lying mostly inside an ignore-region are
neither true positives nor false positives — they are excluded and counted
separately, so an unresolved droplet cannot inflate or deflate the score.

In [ ]:
def _iou_matrix(gt_lbl: np.ndarray, pred_lbl: np.ndarray) -> np.ndarray:
    """(n_gt+1, n_pred+1) IoU table, computed from one contingency pass."""
    n_g, n_p = int(gt_lbl.max()), int(pred_lbl.max())
    if n_g == 0 or n_p == 0:
        return np.zeros((n_g + 1, n_p + 1), np.float64)
    flat = (gt_lbl.astype(np.int64) * (n_p + 1) + pred_lbl.astype(np.int64)).ravel()
    inter = np.bincount(flat, minlength=(n_g + 1) * (n_p + 1)
                        ).reshape(n_g + 1, n_p + 1).astype(np.float64)
    a_g = inter.sum(axis=1, keepdims=True)
    a_p = inter.sum(axis=0, keepdims=True)
    union = a_g + a_p - inter
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(union > 0, inter / union, 0.0)


def evaluate_instances(gt_mask: np.ndarray, pred_mask: np.ndarray,
                       ignore: Optional[np.ndarray], cfg: GoldStandardConfig) -> dict:
    """Instance-level agreement, with splits, merges and area bias."""
    ignore = (np.zeros_like(gt_mask, bool) if ignore is None else ignore.astype(bool))
    gt_lbl = measure.label(gt_mask & ~ignore, connectivity=2)
    pred_lbl_all = measure.label(pred_mask, connectivity=2)

    # drop predicted objects sitting mostly in ignore-regions
    excluded = 0
    pred_lbl = pred_lbl_all.copy()
    for r in measure.regionprops(pred_lbl_all):
        if ignore[tuple(r.coords.T)].mean() > 0.5:
            pred_lbl[pred_lbl_all == r.label] = 0
            excluded += 1
    pred_lbl = measure.label(pred_lbl > 0, connectivity=2)

    iou = _iou_matrix(gt_lbl, pred_lbl)
    n_g, n_p = int(gt_lbl.max()), int(pred_lbl.max())
    if n_g == 0 or n_p == 0:
        return {"n_gt": n_g, "n_pred": n_p, "tp": 0, "fp": n_p, "fn": n_g,
                "precision": np.nan, "recall": np.nan, "f1": np.nan,
                "split_rate": np.nan, "merge_rate": np.nan,
                "median_area_bias": np.nan, "mean_matched_iou": np.nan,
                "pred_excluded_by_ignore": excluded}

    core = iou[1:, 1:]
    # greedy one-to-one matching, highest IoU first
    order = np.dstack(np.unravel_index(np.argsort(core, axis=None)[::-1], core.shape))[0]
    g_taken, p_taken, matches = set(), set(), []
    for g, p in order:
        if core[g, p] < cfg.eval_match_iou:
            break
        if g in g_taken or p in p_taken:
            continue
        g_taken.add(g); p_taken.add(p); matches.append((g, p, core[g, p]))

    touch = core >= cfg.eval_touch_iou
    splits = int((touch.sum(axis=1) >= 2).sum())      # one GT, several predictions
    merges = int((touch.sum(axis=0) >= 2).sum())      # one prediction, several GT

    gt_areas = np.bincount(gt_lbl.ravel(), minlength=n_g + 1)[1:]
    pr_areas = np.bincount(pred_lbl.ravel(), minlength=n_p + 1)[1:]
    bias = [(pr_areas[p] - gt_areas[g]) / gt_areas[g] for g, p, _ in matches
            if gt_areas[g] > 0]

    tp = len(matches)
    return {"n_gt": n_g, "n_pred": n_p, "tp": tp, "fp": n_p - tp, "fn": n_g - tp,
            "precision": tp / n_p if n_p else np.nan,
            "recall": tp / n_g if n_g else np.nan,
            "f1": 2 * tp / (n_g + n_p) if (n_g + n_p) else np.nan,
            "split_rate": splits / n_g if n_g else np.nan,
            "merge_rate": merges / n_p if n_p else np.nan,
            "median_area_bias": float(np.median(bias)) if bias else np.nan,
            "mean_matched_iou": float(np.mean([m[2] for m in matches])) if matches else np.nan,
            "pred_excluded_by_ignore": excluded}


def evaluate_semantic(gt_mask, pred_mask, ignore=None) -> dict:
    valid = ~(np.zeros_like(gt_mask, bool) if ignore is None else ignore.astype(bool))
    g, p = gt_mask[valid].astype(bool), pred_mask[valid].astype(bool)
    inter, union = int((g & p).sum()), int((g | p).sum())
    tot = int(g.sum()) + int(p.sum())
    return {"dice": 2 * inter / tot if tot else np.nan,
            "iou": inter / union if union else np.nan,
            "gt_px": int(g.sum()), "pred_px": int(p.sum())}


def boundary_f1(gt_mask, pred_mask, tol_px: int = 2, ignore=None) -> float:
    """F1 over boundary pixels within `tol_px` of the other contour."""
    valid = ~(np.zeros_like(gt_mask, bool) if ignore is None else ignore.astype(bool))
    gb = gt_mask & ~morphology.binary_erosion(gt_mask, morphology.disk(1)) & valid
    pb = pred_mask & ~morphology.binary_erosion(pred_mask, morphology.disk(1)) & valid
    if not gb.any() or not pb.any():
        return float("nan")
    d_g = ndi.distance_transform_edt(~gb)
    d_p = ndi.distance_transform_edt(~pb)
    prec = float((d_g[pb] <= tol_px).mean())
    rec = float((d_p[gb] <= tol_px).mean())
    return 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0


def evaluate_against_gold_standard(pred_by_sample: Dict[str, np.ndarray],
                                   gs_paths: GoldStandardPaths,
                                   cfg: GoldStandardConfig,
                                   class_index: int = CLASS_NUCLEUS) -> pd.DataFrame:
    """
    pred_by_sample: {sample_id: predicted binary mask (H, W) for `class_index`}.
    Produce those with Vulcan 1.1 at the committed planes' (t, z) — the model
    call itself stays in the pipeline notebook so this one never loads TensorFlow.
    """
    rows = []
    for c in load_committed(gs_paths):
        sid = c["meta"]["sample_id"]
        if sid not in pred_by_sample:
            continue
        gt_ch = c["labels"][class_index]
        ignore = gt_ch == UNANNOTATED
        gt = gt_ch == 1
        pred = np.asarray(pred_by_sample[sid]).astype(bool)
        if pred.shape != gt.shape:
            raise ValueError(f"{sid}: prediction {pred.shape} vs GT {gt.shape}")
        row = {"sample_id": sid, "t": c["meta"]["t"], "z": c["meta"]["z"],
               "class": CLASS_NAMES[class_index],
               "ignore_frac": float(ignore.mean())}
        row.update(evaluate_semantic(gt, pred, ignore))
        row.update(evaluate_instances(gt, pred, ignore, cfg))
        row["boundary_f1_2px"] = boundary_f1(gt, pred, tol_px=2, ignore=ignore)
        rows.append(row)
    return pd.DataFrame(rows)


def failure_panel(results: pd.DataFrame) -> pd.DataFrame:
    """The four numbers to read first, by timepoint."""
    if results.empty:
        return results
    g = results.groupby("t")
    return pd.DataFrame({
        "planes": g.size(),
        "dice": g.dice.median(),
        "f1@0.5": g.f1.median(),
        "split_rate": g.split_rate.median(),      # fragmentation
        "merge_rate": g.merge_rate.median(),
        "area_bias": g.median_area_bias.median(),  # incomplete extent
        "boundary_f1": g.boundary_f1_2px.median(),
        "ignore_frac": g.ignore_frac.median(),
    }).round(3)

### 13b. Self-test of the metrics

The metrics are the instrument. Before trusting a number they produce about
Vulcan, check they respond correctly to a *known* failure: take a ground-truth
nucleus and break it into three fragments covering 60 % of the area. Dice
barely moves; `split_rate` and `area_bias` should both fire.

In [ ]:
def metrics_self_test(cfg: GoldStandardConfig) -> pd.DataFrame:
    H = W = 200
    yy, xx = np.mgrid[:H, :W]
    gt = np.zeros((H, W), bool)
    for cy, cx in ((60, 60), (60, 140), (140, 100)):
        gt |= (yy - cy) ** 2 + (xx - cx) ** 2 <= 25 ** 2

    perfect = gt.copy()

    fragmented = np.zeros_like(gt)                       # 1 nucleus → 3 pieces
    for cy, cx in ((60, 60), (60, 140), (140, 100)):
        for dy, dx in ((-11, 0), (11, -9), (11, 9)):
            fragmented |= (yy - cy - dy) ** 2 + (xx - cx - dx) ** 2 <= 9 ** 2

    shrunk = morphology.binary_erosion(gt, morphology.disk(5))   # correct count, small

    rows = []
    for name, pred in (("perfect", perfect), ("fragmented", fragmented),
                       ("shrunk", shrunk)):
        r = {"case": name}
        r.update(evaluate_semantic(gt, pred))
        r.update(evaluate_instances(gt, pred, None, cfg))
        rows.append(r)
    return pd.DataFrame(rows)[
        ["case", "dice", "n_pred", "tp", "f1", "split_rate", "median_area_bias"]]


st = metrics_self_test(cfg)
print(st.round(3).to_string(index=False))
assert st.loc[st.case == "fragmented", "split_rate"].iloc[0] > 0, \
    "split_rate failed to detect fragmentation"
assert st.loc[st.case == "shrunk", "median_area_bias"].iloc[0] < -0.2, \
    "area_bias failed to detect under-extended masks"
assert st.loc[st.case == "fragmented", "dice"].iloc[0] > 0.5, \
    "the fragmented case should still score respectably on Dice — that is the point"
print("\nmetrics respond correctly to both failure modes.")

## 14. Order of operations

1. **§7** — run the synthetic self-test. It needs no data and fails loudly.
2. **§8** — `RUN_AUDIT = True`. Read `legacy_fragments` and
   `area_ratio_ws_over_legacy` per timepoint. If fragmentation is *not* visible
   in the audit, the hypothesis is wrong for this dataset and the rest of the
   notebook should not be run on the strength of the synthetic result alone.
3. Tune `nuc_flatten_um` on the gallery in §8, if needed. It is the one knob
   with real leverage; the rest are gates.
4. **§9–10** — freeze the plan, `RUN_PROPOSALS = True`. Proposals are written
   once; the plan cannot be edited afterwards without a new run id.
5. **§11** — review in `napari_env`, plane by plane.
6. **§12** — commit. Rejected planes go back to review; forcing a commit is
   recorded in the manifest.
7. **§13** — run Vulcan 1.1 over the committed planes in the pipeline notebook,
   pass `{sample_id: nucleus_mask}` to `evaluate_against_gold_standard`, read
   `failure_panel`.

### Open items

* **`gate_min_timepoint = 2`** means every t = 0–1 droplet is written as
  `UNANNOTATED`, so early timepoints will be human-annotated or excluded.
  That is the honest handling — the Stage-2 gate carries no information there —
  but it does mean the early phase costs the most reviewer time.
* **One nucleus per droplet** (`nuc_max_seeds = 1`) is a hard prior. Droplets
  that genuinely contain two nuclei will be under-segmented, and the existing
  `flag_close_nuclei` exclusion suggests they exist. Raising `nuc_max_seeds`
  makes the watershed split them correctly; the reason to leave it at 1 is that
  a spurious second seed then produces a spurious second nucleus. Worth
  revisiting once the audit shows how often it happens.
* **`nuc_flatten_um` is fixed, and adapting it is harder than it looks.**
  The §7b sweep shows the cliff: at the 4.0 µm default, nuclei of radius
  ≥ 4.5 µm score ≥ 0.96, and a 3 µm nucleus scores 0.40 — the structuring
  element erodes what it is meant to preserve. A 3 µm radius is 28 µm², below
  this dataset's 100 µm² floor, so the cliff sits outside the real range; the
  closest real case (64 µm², r = 4.5 µm) scores 0.96.

  **A two-pass version was tried and does not work.** Segment at 4.0 µm,
  estimate r from the result, re-flatten at 0.6 × r, re-segment: measured
  0.899 vs 0.983 single-pass at r = 6 µm, 0.757 vs 0.958 at r = 4.5 µm, and
  at r = 3 µm it diverges to the 6 µm ceiling. The feedback amplifies a bad
  first pass instead of correcting it — an under-segmented result asks for a
  smaller SE, which under-segments further. If small nuclei do turn out to be
  a problem in the §8 audit, the fix is a *seed-size* prior or an
  NPC-ring-radius estimate as an independent size input, not a feedback loop
  on the segmentation's own output.

* **Do not reintroduce a droplet-relative clamp** (§7c). It biased the
  measured area of a fixed nucleus by −54 % in a 200 µm² droplet against −2 %
  above 500 µm², i.e. it manufactured the nuclear-size/droplet-size
  correlation this project measures. `guard_no_droplet_size_bias` fails if any
  droplet-size-dependent term comes back.
* **Vulcan inherits the label generator's fragmentation.** If the audit
  confirms this failure in the labels, the training set has it too, and the
  gold standard will show Vulcan reproducing it. The fix is then to regenerate
  training labels with §5 and retrain, not to adjust inference thresholds.